# Rust Duty · animation review / Colab
Status: diagnostic. Technical measurements never grant artistic approval.

1. Open this notebook in Colab. Runtime → Change runtime type → A100 GPU, if available. CPU is sufficient for pack checks/plots. An A100 only runs the optional explicitly enabled contact-array calculation; no ML model, training, video rendering or Blender installation is performed.
2. Run setup, then the synthetic demo once. Set DEMO=False and upload a ZIP containing only authorized `.vra` and optional matching `.vrs`/`.vrm`, JSON telemetry, and source/candidate PNG/JPEG stills. Never upload private soldier assets. You choose inputs; this notebook does not clone repositories, mount Drive, request credentials, or send files anywhere else.
3. Select a pack, clip, channel and explicit transitions. Run the review and download the results ZIP.
4. Save the results before Runtime → Disconnect and delete runtime. Do not leave A100 allocated for CPU work. Availability, runtime limits and compute-unit consumption vary; no fixed cost promise is made.

[Official Colab FAQ](https://research.google.com/colaboratory/faq.html). Notebook sharing includes saved outputs: clear them before sharing.

## Scope
VRANIM01 validated with the project's embedded original parsers. Companion checks verify structure/checksums/bone identity/rigid assignments when companions exist, not evaluated skin or contact. Curves are stored parent-local bone TRS and scene-global actor TRS, glTF Y-up. Runtime applies Ry(pi); runtime blend logic is not reproduced. Translation units require source verification. Loop endpoints and selected end-to-start joins are diagnostics, never an automatic failure threshold. Speed statistics are sampled segment-average values; shortest-arc rotations can alias motion over 180 degrees between samples. They are not continuous-time peak speeds.


In [ ]:
# Pinned direct dependencies from the standard PyPI registry. No shell installer scripts.
%pip -q install numpy==2.2.6 matplotlib==3.10.8 Pillow==12.3.0
# NumPy 2.2.6 satisfies Colab's observed Numba 0.61.2 (<2.3) constraint.
# Do not analyze with an older NumPy module still loaded after changing its wheel.
import sys, importlib.metadata
installed_numpy=importlib.metadata.version('numpy')
loaded_numpy=getattr(sys.modules.get('numpy'),'__version__',installed_numpy)
if installed_numpy!='2.2.6': raise RuntimeError('NumPy installation did not match the 2.2.6 pin')
if loaded_numpy!=installed_numpy:
    raise RuntimeError('NumPy changed while already imported. Restart the session, then rerun from setup before analysis.')
try:
    from packaging.requirements import Requirement
    for item in importlib.metadata.requires('numba') or []:
        req=Requirement(item)
        if req.name.lower()=='numpy' and (req.marker is None or req.marker.evaluate()):
            if installed_numpy not in req.specifier:
                raise RuntimeError('Installed Numba requires '+str(req)+'; stop before analysis')
except importlib.metadata.PackageNotFoundError:
    pass # Numba is optional and not installed by this notebook.


In [ ]:
# Frozen original source modules; no remote code fetch and no uploaded code execution.
import sys, types, hashlib, json, platform, tempfile, zipfile, importlib.metadata
from pathlib import Path
SOURCES = {'vrpack': '#!/usr/bin/env python3\n"""Strict local GLB 2.0 -> Vector Range mesh converter. See docs/ASSET_FORMAT.md.\n\nNo source GLB, JSON metadata, names, animation, or external resources are copied.\nThe format is not encrypted and provides no protection against extraction.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport struct\nimport sys\nimport tempfile\nimport zlib\n\nMAGIC = b"VRMESH01"\nVERSION = 1\nHEADER = struct.Struct("<8sIIII")\nMAX_FILE = 128 * 1024 * 1024\nMAX_MESHES = 256\nMAX_VERTICES = 1_000_000\nMAX_INDICES = 3_000_000\nMAX_TEXTURE_DIM = 2048\nMAX_TEXTURE_BYTES = 64 * 1024 * 1024\nMAX_NODES = 100_000\nMAX_DEPTH = 256\nMAX_POSITION = 10_000.0\nMAX_UV = 1_000_000.0\nIDENTITY = (1.0, 0.0, 0.0, 0.0,\n            0.0, 1.0, 0.0, 0.0,\n            0.0, 0.0, 1.0, 0.0,\n            0.0, 0.0, 0.0, 1.0)\n\n\nclass AssetError(ValueError):\n    """Invalid or unsupported source/output; safe to display as a CLI error."""\n\n\ndef require(condition, message):\n    if not condition:\n        raise AssetError(message)\n\n\ndef uint(value, label, maximum=0xFFFFFFFF):\n    require(type(value) is int and 0 <= value <= maximum,\n            f"{label} must be a nonnegative integer <= {maximum}")\n    return value\n\n\ndef number(value, label):\n    require(type(value) in (int, float), f"{label} must be a number")\n    try:\n        value = float(value)\n    except (ValueError, OverflowError) as exc:\n        raise AssetError(f"{label} must be finite") from exc\n    require(math.isfinite(value) and abs(value) <= 3.4028234663852886e38,\n            f"{label} must be finite and representable as float32")\n    return value\n\n\ndef vector(value, size, label):\n    require(isinstance(value, list) and len(value) == size,\n            f"{label} must have {size} numbers")\n    return tuple(number(x, label) for x in value)\n\n\ndef objects(value, label):\n    require(isinstance(value, list) and all(isinstance(x, dict) for x in value),\n            f"{label} must be an array of objects")\n    return value\n\n\ndef at(items, index, label):\n    index = uint(index, label)\n    require(index < len(items), f"{label} is out of range")\n    return items[index]\n\n\ndef read_limited(path):\n    with open(path, "rb") as source:\n        data = source.read(MAX_FILE + 1)\n    require(len(data) <= MAX_FILE, "file exceeds 128 MiB limit")\n    return data\n\n\ndef _no_duplicate_keys(pairs):\n    result = {}\n    for key, value in pairs:\n        require(key not in result, f"duplicate JSON key: {key}")\n        result[key] = value\n    return result\n\n\ndef _bad_constant(value):\n    raise AssetError(f"non-finite JSON constant: {value}")\n\n\ndef parse_glb(data):\n    require(12 <= len(data) <= MAX_FILE, "GLB is truncated or exceeds 128 MiB")\n    magic, version, length = struct.unpack_from("<4sII", data)\n    require(magic == b"glTF", "input is not a GLB")\n    require(version == 2, "only GLB version 2 is supported")\n    require(length == len(data), "GLB declared length does not match file")\n    chunks = []\n    cursor = 12\n    while cursor < length:\n        require(len(chunks) < 2, "GLB contains more than JSON and BIN chunks")\n        require(cursor + 8 <= length, "truncated GLB chunk header")\n        size, kind = struct.unpack_from("<II", data, cursor)\n        cursor += 8\n        require(size % 4 == 0, "GLB chunk length must be 4-byte aligned")\n        require(size <= length - cursor, "GLB chunk exceeds file")\n        chunks.append((kind, data[cursor:cursor + size]))\n        cursor += size\n    require(1 <= len(chunks) <= 2 and chunks[0][0] == 0x4E4F534A,\n            "GLB must contain JSON first, followed only by optional BIN")\n    require(len(chunks) == 1 or chunks[1][0] == 0x004E4942,\n            "unsupported GLB chunk (only JSON and BIN are accepted)")\n    try:\n        doc = json.loads(chunks[0][1].decode("utf-8"),\n                         parse_constant=_bad_constant,\n                         object_pairs_hook=_no_duplicate_keys)\n    except (UnicodeDecodeError, ValueError, RecursionError) as exc:\n        raise AssetError(f"invalid GLB JSON: {exc}") from exc\n    require(isinstance(doc, dict), "GLB JSON root must be an object")\n    asset = doc.get("asset", {})\n    require(isinstance(asset, dict) and asset.get("version") == "2.0",\n            "only glTF asset version 2.0 is supported")\n    require(asset.get("minVersion", "2.0") == "2.0", "unsupported glTF minVersion")\n    return doc, chunks[1][1] if len(chunks) == 2 else b""\n\n\ndef matmul(a, b):\n    """Row-major internal matrices; glTF input matrices are column-major."""\n    return tuple(sum(a[r * 4 + k] * b[k * 4 + c] for k in range(4))\n                 for r in range(4) for c in range(4))\n\n\ndef determinant(m):\n    a, b, c, d, e, f, g, h, i = (m[x] for x in (0, 1, 2, 4, 5, 6, 8, 9, 10))\n    return a * (e * i - f * h) - b * (d * i - f * g) + c * (d * h - e * g)\n\n\ndef normal_matrix(m):\n    a, b, c, d, e, f, g, h, i = (m[x] for x in (0, 1, 2, 4, 5, 6, 8, 9, 10))\n    det = determinant(m)\n    require(math.isfinite(det) and det != 0.0, "singular or non-finite node transform")\n    # Cofactor matrix divided by determinant = inverse transpose.\n    return tuple(x / det for x in (e*i-f*h, f*g-d*i, d*h-e*g,\n                                   c*h-b*i, a*i-c*g, b*g-a*h,\n                                   b*f-c*e, c*d-a*f, a*e-b*d))\n\n\ndef f32(value):\n    """Round a bounded arithmetic result exactly as the runtime float type."""\n    return struct.unpack("<f", struct.pack("<f", value))[0]\n\n\ndef normalized(v, label):\n    length = math.hypot(*v)\n    require(math.isfinite(length) and length > 0.0, f"{label} has zero/non-finite length")\n    result = tuple(x / length for x in v)\n    require(all(math.isfinite(x) for x in result), f"{label} is non-finite")\n    return result\n\n\ndef node_matrix(node):\n    if "matrix" in node:\n        require(not any(k in node for k in ("translation", "rotation", "scale")),\n                "node cannot combine matrix with TRS")\n        col = vector(node["matrix"], 16, "node matrix")\n        result = tuple(col[c * 4 + r] for r in range(4) for c in range(4))\n        require(result[12:16] == (0.0, 0.0, 0.0, 1.0), "node matrix must be affine")\n        return result\n    t = vector(node.get("translation", [0, 0, 0]), 3, "node translation")\n    s = vector(node.get("scale", [1, 1, 1]), 3, "node scale")\n    q = vector(node.get("rotation", [0, 0, 0, 1]), 4, "node rotation")\n    require(abs(math.hypot(*q) - 1.0) <= 0.001, "node quaternion must be normalized")\n    x, y, z, w = normalized(q, "node quaternion")\n    rot = (1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w), 0,\n           2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w), 0,\n           2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y), 0,\n           0, 0, 0, 1)\n    return (rot[0]*s[0], rot[1]*s[1], rot[2]*s[2], t[0],\n            rot[4]*s[0], rot[5]*s[1], rot[6]*s[2], t[1],\n            rot[8]*s[0], rot[9]*s[1], rot[10]*s[2], t[2],\n            0.0, 0.0, 0.0, 1.0)\n\n\ndef reject_extensions(doc):\n    # Extras are intentionally ignored. Extensions may change the interpretation\n    # of otherwise ordinary values, so even optional extensions are rejected.\n    stack = [doc]\n    while stack:\n        value = stack.pop()\n        if isinstance(value, dict):\n            for key in ("extensions", "extensionsUsed", "extensionsRequired"):\n                require(not value.get(key), f"glTF {key} are not supported")\n            stack.extend(v for k, v in value.items() if k != "extras")\n        elif isinstance(value, list):\n            stack.extend(value)\n\n\nclass Converter:\n    def __init__(self, doc, blob):\n        self.doc, self.blob = doc, blob\n        reject_extensions(doc)\n        require(not doc.get("animations") and not doc.get("skins"),\n                "animations and skinning are not supported; export a static mesh")\n        self.buffers = objects(doc.get("buffers", []), "buffers")\n        require(len(self.buffers) == 1, "one embedded GLB buffer is required")\n        require("uri" not in self.buffers[0], "external/data-URI buffers are not supported")\n        self.buffer_length = uint(self.buffers[0].get("byteLength"), "buffer byteLength")\n        require(self.buffer_length <= len(blob) <= self.buffer_length + 3,\n                "BIN length does not match buffer byteLength (plus <=3 padding bytes)")\n        require(not any(blob[self.buffer_length:]), "BIN padding must be zero")\n        self.views = objects(doc.get("bufferViews", []), "bufferViews")\n        self.accessors = objects(doc.get("accessors", []), "accessors")\n        self.images = objects(doc.get("images", []), "images")\n        self.textures = objects(doc.get("textures", []), "textures")\n        self.samplers = objects(doc.get("samplers", []), "samplers")\n        self.materials = objects(doc.get("materials", []), "materials")\n        self.meshes = objects(doc.get("meshes", []), "meshes")\n        self.nodes = objects(doc.get("nodes", []), "nodes")\n        self.scenes = objects(doc.get("scenes", []), "scenes")\n        require(len(self.nodes) <= MAX_NODES, "too many GLB nodes")\n        for img in self.images:\n            require("uri" not in img, "external/data-URI images are not supported")\n        for mesh in self.meshes:\n            require("weights" not in mesh, "morph weights are not supported")\n            for primitive in objects(mesh.get("primitives", []), "primitives"):\n                require("targets" not in primitive, "morph targets are not supported")\n        for node in self.nodes:\n            require("skin" not in node and "weights" not in node,\n                    "node skinning/morph weights are not supported")\n        for material in self.materials:\n            self.validate_material(material)\n        self.total_vertices = self.total_indices = self.total_texels = 0\n        self.texture_cache = {}\n\n    def view(self, index):\n        view = at(self.views, index, "bufferView")\n        require(view.get("buffer") == 0 and type(view.get("buffer")) is int,\n                "bufferView must reference embedded buffer 0")\n        offset = uint(view.get("byteOffset", 0), "bufferView byteOffset")\n        length = uint(view.get("byteLength"), "bufferView byteLength")\n        require(offset <= self.buffer_length and length <= self.buffer_length - offset,\n                "bufferView exceeds embedded buffer")\n        return view, offset, length\n\n    def accessor(self, index, semantic):\n        acc = at(self.accessors, index, f"{semantic} accessor")\n        require("sparse" not in acc, "sparse accessors are not supported")\n        require(acc.get("normalized", False) is False, "normalized accessors are not supported")\n        if semantic == "indices":\n            comp = acc.get("componentType")\n            require(type(comp) is int and comp in (5121, 5123, 5125),\n                    "indices require UNSIGNED_BYTE/SHORT/INT")\n            components, fmt, size, maximum = 1, {5121: "B", 5123: "H", 5125: "I"}[comp], {5121: 1, 5123: 2, 5125: 4}[comp], MAX_INDICES\n            require(acc.get("type") == "SCALAR", "indices require SCALAR accessor")\n        else:\n            components = 2 if semantic == "TEXCOORD_0" else 3\n            require(acc.get("componentType") == 5126 and type(acc.get("componentType")) is int,\n                    f"{semantic} requires FLOAT components")\n            require(acc.get("type") == f"VEC{components}", f"{semantic} requires VEC{components}")\n            fmt, size, maximum = "f", 4, MAX_VERTICES\n        count = uint(acc.get("count"), f"{semantic} count", maximum)\n        require(count > 0, f"{semantic} accessor is empty")\n        view, base, length = self.view(acc.get("bufferView"))\n        offset = uint(acc.get("byteOffset", 0), "accessor byteOffset")\n        require(offset % size == 0 and (base + offset) % size == 0,\n                "accessor offset is not component aligned")\n        elem_size = size * components\n        if "byteStride" in view:\n            require(semantic != "indices", "indices cannot have byteStride")\n            stride = uint(view["byteStride"], "byteStride", 252)\n            require(stride >= elem_size and stride % 4 == 0, "invalid accessor byteStride")\n        else:\n            stride = elem_size\n        require(offset <= length and (count - 1) * stride + elem_size <= length - offset,\n                "accessor exceeds bufferView bounds")\n        unpack = struct.Struct("<" + fmt * components).unpack_from\n        result = [unpack(self.blob, base + offset + i * stride) for i in range(count)]\n        if semantic != "indices":\n            require(all(math.isfinite(x) for row in result for x in row),\n                    f"{semantic} contains non-finite floats")\n        return [x[0] for x in result] if semantic == "indices" else result\n\n    @staticmethod\n    def validate_material(material):\n        require(material.get("alphaMode", "OPAQUE") == "OPAQUE", "only OPAQUE materials are supported")\n        require(material.get("doubleSided", False) is False, "double-sided materials are not supported")\n        for field in ("normalTexture", "occlusionTexture", "emissiveTexture"):\n            require(field not in material, f"{field} is not supported")\n        emissive = vector(material.get("emissiveFactor", [0, 0, 0]), 3, "emissiveFactor")\n        require(emissive == (0, 0, 0), "emissive material factors are not supported")\n        pbr = material.get("pbrMetallicRoughness", {})\n        require(isinstance(pbr, dict), "pbrMetallicRoughness must be an object")\n        require("metallicRoughnessTexture" not in pbr, "metallicRoughnessTexture is not supported")\n        color = vector(pbr.get("baseColorFactor", [1, 1, 1, 1]), 4, "baseColorFactor")\n        factors = color + (number(pbr.get("metallicFactor", 1), "metallicFactor"),\n                           number(pbr.get("roughnessFactor", 1), "roughnessFactor"))\n        require(all(0 <= f <= 1 for f in factors), "material factors must be in [0, 1]")\n        return pbr, factors\n\n    def texture(self, texture_info):\n        require(isinstance(texture_info, dict), "baseColorTexture must be an object")\n        require(texture_info.get("texCoord", 0) == 0 and type(texture_info.get("texCoord", 0)) is int,\n                "only TEXCOORD_0 is supported")\n        index = uint(texture_info.get("index"), "texture index")\n        if index in self.texture_cache:\n            return self.texture_cache[index]\n        texture = at(self.textures, index, "texture")\n        if "sampler" in texture:\n            sampler = at(self.samplers, texture["sampler"], "sampler")\n            require(sampler.get("wrapS", 10497) == 10497 and sampler.get("wrapT", 10497) == 10497,\n                    "v1 supports only REPEAT texture wrapping")\n            require(sampler.get("magFilter", 9729) == 9729 and sampler.get("minFilter", 9729) == 9729,\n                    "v1 supports only LINEAR texture filtering without mipmaps")\n        image = at(self.images, texture.get("source"), "texture source")\n        mime = image.get("mimeType")\n        require(mime in ("image/png", "image/jpeg"), "only embedded PNG/JPEG images are supported")\n        view, offset, length = self.view(image.get("bufferView"))\n        require("byteStride" not in view, "image bufferView cannot have a byteStride")\n        try:\n            from PIL import Image\n        except ImportError as exc:\n            raise AssetError("embedded textures need Pillow: python3 -m pip install -r tools/requirements-assets.txt") from exc\n        try:\n            with Image.open(io.BytesIO(self.blob[offset:offset + length])) as decoded:\n                require(decoded.format == {"image/png": "PNG", "image/jpeg": "JPEG"}[mime],\n                        "image bytes do not match mimeType")\n                width, height = decoded.size\n                require(0 < width <= MAX_TEXTURE_DIM and 0 < height <= MAX_TEXTURE_DIM,\n                        "texture dimensions must be 1..2048 (resize before packing)")\n                require(getattr(decoded, "n_frames", 1) == 1, "animated images are not supported")\n                # Check dimensions before decoding or allocating the RGBA raster.\n                raster = decoded.convert("RGBA").tobytes()\n        except (OSError, ValueError, Image.DecompressionBombError) as exc:\n            if isinstance(exc, AssetError):\n                raise\n            raise AssetError(f"cannot decode embedded image: {exc}") from exc\n        require(len(raster) == width * height * 4, "decoded texture length mismatch")\n        result = width, height, raster\n        self.texture_cache[index] = result\n        return result\n\n    def primitive(self, primitive, world):\n        require(primitive.get("mode", 4) == 4 and type(primitive.get("mode", 4)) is int,\n                "only TRIANGLES primitives are supported")\n        attrs = primitive.get("attributes")\n        require(isinstance(attrs, dict) and "POSITION" in attrs, "primitive requires POSITION")\n        unsupported = set(attrs) - {"POSITION", "NORMAL", "TEXCOORD_0"}\n        require(not unsupported, f"unsupported vertex attributes: {\', \'.join(sorted(unsupported))}")\n        positions = self.accessor(attrs["POSITION"], "POSITION")\n        n = len(positions)\n        indices = self.accessor(primitive["indices"], "indices") if "indices" in primitive else list(range(n))\n        require(len(indices) % 3 == 0, "triangle index count must be divisible by 3")\n        require(all(i < n for i in indices), "index references a vertex outside POSITION")\n        self.total_vertices += n\n        self.total_indices += len(indices)\n        require(self.total_vertices <= MAX_VERTICES and self.total_indices <= MAX_INDICES,\n                "aggregate vertex/index limit exceeded")\n        if "NORMAL" in attrs:\n            normals = self.accessor(attrs["NORMAL"], "NORMAL")\n            require(len(normals) == n, "NORMAL count must match POSITION")\n            normals = [normalized(v, "source normal") for v in normals]\n        else:\n            normals = [[0.0, 0.0, 0.0] for _ in positions]\n            for k in range(0, len(indices), 3):\n                ia, ib, ic = indices[k:k + 3]\n                a, b, c = positions[ia], positions[ib], positions[ic]\n                u = tuple(b[j] - a[j] for j in range(3))\n                v = tuple(c[j] - a[j] for j in range(3))\n                cross = (u[1]*v[2]-u[2]*v[1], u[2]*v[0]-u[0]*v[2], u[0]*v[1]-u[1]*v[0])\n                for idx in (ia, ib, ic):\n                    for j in range(3):\n                        normals[idx][j] += cross[j]\n            normals = [normalized(v, "computed normal (isolated/degenerate vertex)") for v in normals]\n        uvs = self.accessor(attrs["TEXCOORD_0"], "TEXCOORD_0") if "TEXCOORD_0" in attrs else [(0.0, 0.0)] * n\n        require(len(uvs) == n, "TEXCOORD_0 count must match POSITION")\n        material = at(self.materials, primitive["material"], "material") if "material" in primitive else {}\n        pbr, factors = self.validate_material(material)\n        width, height, raster = 0, 0, b""\n        if "baseColorTexture" in pbr:\n            require("TEXCOORD_0" in attrs, "textured primitive requires TEXCOORD_0")\n            width, height, raster = self.texture(pbr["baseColorTexture"])\n        self.total_texels += len(raster)\n        require(self.total_texels <= MAX_TEXTURE_BYTES, "aggregate texture bytes exceed 64 MiB")\n        nm = normal_matrix(world)\n        vertices = []\n        for pos, normal, uv in zip(positions, normals, uvs):\n            transformed = tuple(sum(world[r * 4 + j] * pos[j] for j in range(3)) + world[r * 4 + 3] for r in range(3))\n            direction = normalized(tuple(sum(nm[r * 3 + j] * normal[j] for j in range(3)) for r in range(3)), "transformed normal")\n            vertex = transformed + direction + tuple(uv)\n            require(all(abs(v) <= MAX_POSITION for v in transformed), "output position exceeds +/-10000 units")\n            require(all(abs(v) <= MAX_UV for v in uv), "output UV exceeds +/-1000000")\n            for value in vertex:\n                number(value, "output vertex")\n            vertices.append(vertex)\n        if determinant(world) < 0:\n            for k in range(0, len(indices), 3):\n                indices[k + 1], indices[k + 2] = indices[k + 2], indices[k + 1]\n        return factors, width, height, raster, vertices, indices\n\n    def convert(self, scale=1.0, axes="gltf", translate=(0.0, 0.0, 0.0)):\n        scale = number(scale, "scale")\n        require(scale > 0, "scale must be positive")\n        require(axes in ("gltf", "z-up"), "axes must be gltf or z-up")\n        shift = vector(list(translate), 3, "translate")\n        root_transform = (scale, 0, 0, 0, 0, scale, 0, 0, 0, 0, scale, 0, 0, 0, 0, 1)\n        if axes == "z-up":\n            root_transform = (scale, 0, 0, 0, 0, 0, scale, 0, 0, -scale, 0, 0, 0, 0, 0, 1)\n        root_transform = list(root_transform)\n        root_transform[3], root_transform[7], root_transform[11] = shift\n        root_transform = tuple(root_transform)\n        require(self.scenes, "a scene with mesh nodes is required")\n        scene = at(self.scenes, self.doc.get("scene", 0), "default scene")\n        roots = scene.get("nodes", [])\n        require(isinstance(roots, list), "scene nodes must be an array")\n        require(len(roots) == len(set(uint(n, "scene node") for n in roots)), "duplicate root nodes")\n        seen, active = set(), set()\n        output = []\n\n        def walk(index, parent, depth):\n            require(depth <= MAX_DEPTH, "node hierarchy exceeds 256 levels")\n            node = at(self.nodes, index, "node")\n            require(index not in active, "cycle in node hierarchy")\n            require(index not in seen, "node has multiple parents or occurs multiple times")\n            seen.add(index)\n            active.add(index)\n            world = matmul(parent, node_matrix(node))\n            require(all(math.isfinite(v) for v in world), "non-finite composed transform")\n            if "mesh" in node:\n                mesh = at(self.meshes, node["mesh"], "node mesh")\n                primitives = objects(mesh.get("primitives", []), "primitives")\n                require(primitives, "mesh has no primitives")\n                require(len(output) + len(primitives) <= MAX_MESHES, "mesh primitive count exceeds 256")\n                for primitive in primitives:\n                    output.append(self.primitive(primitive, world))\n            children = node.get("children", [])\n            require(isinstance(children, list), "node children must be an array")\n            for child in children:\n                walk(uint(child, "child node"), world, depth + 1)\n            active.remove(index)\n\n        for root in roots:\n            walk(uint(root, "root node"), root_transform, 0)\n        require(output, "default scene contains no triangle meshes")\n        return encode_vrm(output)\n\n\ndef encode_vrm(meshes):\n    payload = bytearray(struct.pack("<I", len(meshes)))\n    for factors, width, height, raster, vertices, indices in meshes:\n        payload.extend(struct.pack("<6fIII", *factors, width, height, len(raster)))\n        payload.extend(raster)\n        payload.extend(struct.pack("<II", len(vertices), len(indices)))\n        for vertex in vertices:\n            payload.extend(struct.pack("<8f", *vertex))\n        # Chunk to avoid a million-argument struct.pack call.\n        for start in range(0, len(indices), 16384):\n            part = indices[start:start + 16384]\n            payload.extend(struct.pack("<" + "I" * len(part), *part))\n        require(len(payload) + HEADER.size <= MAX_FILE, "output exceeds 128 MiB")\n    data = HEADER.pack(MAGIC, VERSION, len(payload), zlib.crc32(payload), 0) + payload\n    inspect_vrm(data)  # Self-check exact binary contract before committing a file.\n    return data\n\n\ndef inspect_vrm(data):\n    require(HEADER.size <= len(data) <= MAX_FILE, "VRM is truncated or exceeds 128 MiB")\n    magic, version, length, checksum, reserved = HEADER.unpack_from(data)\n    require(magic == MAGIC, "invalid VRM magic")\n    require(version == VERSION, "unsupported VRM version")\n    require(reserved == 0, "VRM reserved header word must be zero")\n    require(length == len(data) - HEADER.size, "VRM payload length mismatch")\n    payload = memoryview(data)[HEADER.size:]\n    require(zlib.crc32(payload) == checksum, "VRM checksum mismatch")\n    cursor = 0\n\n    def unpack(fmt):\n        nonlocal cursor\n        layout = struct.Struct(fmt)\n        require(cursor + layout.size <= len(payload), "truncated VRM payload")\n        result = layout.unpack_from(payload, cursor)\n        cursor += layout.size\n        return result\n\n    count, = unpack("<I")\n    require(0 < count <= MAX_MESHES, "VRM mesh count must be 1..256")\n    total_v = total_i = total_t = 0\n    bounds_min, bounds_max = [math.inf] * 3, [-math.inf] * 3\n    texture_count = 0\n    for _ in range(count):\n        factors = unpack("<6f")\n        require(all(math.isfinite(f) and 0 <= f <= 1 for f in factors), "invalid VRM material factor")\n        width, height, size = unpack("<III")\n        if width == 0 or height == 0 or size == 0:\n            require((width, height, size) == (0, 0, 0), "invalid empty VRM texture")\n        else:\n            require(width <= MAX_TEXTURE_DIM and height <= MAX_TEXTURE_DIM and size == width * height * 4,\n                    "invalid VRM texture dimensions or length")\n            texture_count += 1\n        total_t += size\n        require(total_t <= MAX_TEXTURE_BYTES, "VRM aggregate texture limit exceeded")\n        require(size <= len(payload) - cursor, "truncated VRM texture")\n        cursor += size\n        vertex_count, index_count = unpack("<II")\n        require(vertex_count > 0 and index_count > 0 and index_count % 3 == 0,\n                "VRM needs nonempty triangle geometry")\n        total_v += vertex_count\n        total_i += index_count\n        require(total_v <= MAX_VERTICES and total_i <= MAX_INDICES, "VRM aggregate geometry limit exceeded")\n        require(vertex_count * 32 + index_count * 4 <= len(payload) - cursor,\n                "VRM geometry exceeds payload")\n        for _ in range(vertex_count):\n            vertex = unpack("<8f")\n            require(all(math.isfinite(v) for v in vertex), "VRM vertex contains non-finite float")\n            require(all(abs(v) <= 1.01 for v in vertex[3:6]), "VRM normal is not normalized")\n            # Mirror Rust f32 products/additions and bounds, including rounding\n            # at the tolerance boundary instead of checking Python doubles.\n            n2 = 0.0\n            for value in vertex[3:6]:\n                n2 = f32(n2 + f32(value * value))\n            require(f32(0.98) <= n2 <= f32(1.02), "VRM normal is not normalized")\n            require(all(abs(v) <= MAX_POSITION for v in vertex[:3]), "VRM position exceeds +/-10000 units")\n            require(all(abs(v) <= MAX_UV for v in vertex[6:]), "VRM UV exceeds +/-1000000")\n            for j in range(3):\n                bounds_min[j] = min(bounds_min[j], vertex[j])\n                bounds_max[j] = max(bounds_max[j], vertex[j])\n        for _ in range(index_count):\n            idx, = unpack("<I")\n            require(idx < vertex_count, "VRM index is out of bounds")\n    require(cursor == len(payload), "trailing VRM payload bytes")\n    return {"format": MAGIC.decode("ascii"), "version": version, "file_bytes": len(data),\n            "payload_bytes": length, "crc32": f"{checksum:08x}", "meshes": count,\n            "vertices": total_v, "indices": total_i, "triangles": total_i // 3,\n            "textures": texture_count, "texture_bytes": total_t,\n            "bounds_min": bounds_min, "bounds_max": bounds_max}\n\n\ndef convert_glb(data, scale=1.0, axes="gltf", translate=(0.0, 0.0, 0.0)):\n    doc, blob = parse_glb(data)\n    return Converter(doc, blob).convert(scale=scale, axes=axes, translate=translate)\n\n\ndef atomic_write(path, data, force=False):\n    path = Path(path)\n    require(not path.exists() or force, f"output already exists: {path}; use --force to replace")\n    temp = None\n    try:\n        with tempfile.NamedTemporaryFile(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent, delete=False) as handle:\n            temp = Path(handle.name)\n            handle.write(data)\n            handle.flush()\n            os.fsync(handle.fileno())\n        if force:\n            os.replace(temp, path)\n        else:\n            # Same-directory hard link is an atomic no-clobber publication; the\n            # pre-check alone would race another writer creating the destination.\n            try:\n                os.link(temp, path)\n            except FileExistsError as exc:\n                raise AssetError(f"output already exists: {path}; use --force to replace") from exc\n            temp.unlink()\n        temp = None\n    finally:\n        if temp is not None:\n            temp.unlink(missing_ok=True)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="command", required=True)\n    pack = commands.add_parser("pack", help="convert a locally licensed static GLB to .vrm")\n    pack.add_argument("input", type=Path)\n    pack.add_argument("output", type=Path)\n    pack.add_argument("--scale", type=float, default=1.0, help="positive uniform scale applied after node transforms")\n    pack.add_argument("--axes", choices=("gltf", "z-up"), default="gltf", help="gltf preserves x/y/z; z-up maps (x,y,z) to (x,z,-y)")\n    pack.add_argument("--translate", type=float, nargs=3, metavar=("X", "Y", "Z"), default=(0, 0, 0),\n                      help="translation in output units, applied after scale and axes conversion")\n    pack.add_argument("--force", action="store_true", help="atomically replace an existing output")\n    inspect = commands.add_parser("inspect", help="validate .vrm and print its geometry summary")\n    inspect.add_argument("input", type=Path)\n    args = parser.parse_args(argv)\n    try:\n        source = read_limited(args.input)\n        if args.command == "pack":\n            require(args.input.resolve() != args.output.resolve(), "input and output paths must differ")\n            result = convert_glb(source, scale=args.scale, axes=args.axes, translate=args.translate)\n            atomic_write(args.output, result, args.force)\n            print(f"Packed {args.output}: {json.dumps(inspect_vrm(result), sort_keys=True)}")\n        else:\n            print(json.dumps(inspect_vrm(source), indent=2, sort_keys=True))\n        return 0\n    except (AssetError, OSError, OverflowError, RecursionError, struct.error) as exc:\n        print(f"vrpack: {exc}", file=sys.stderr)\n        return 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n\n', 'vrskin': '#!/usr/bin/env python3\n"""Strict local GLB -> VRSKIN02 converter. The container is not encryption.\n\nUses vrpack\'s unchanged generic GLB/geometry/image validation helpers. VRMESH01\ncontinues to reject all skins; this is a separate, explicitly skinned pipeline.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport json\nimport math\nfrom pathlib import Path\nimport struct\nimport sys\nimport zlib\nimport vrpack as core\nfrom vrpack import AssetError, require, uint, number, objects, at, matmul\n\nMAGIC = b\'VRSKIN02\'\nVERSION = 2\nHEADER = struct.Struct(\'<8sIIII\')\nMAX_BONES = 128\nMAX_MESHES = 128\nVERTEX = struct.Struct(\'<8f8H8f\')\n\n\ndef transpose(m):\n    return tuple(m[c * 4 + r] for r in range(4) for c in range(4))\n\n\ndef matrix_valid(col, label):\n    require(len(col) == 16 and all(math.isfinite(v) and abs(v) <= 3.4028234663852886e38 for v in col),\n            f\'{label} must contain 16 finite float32 values\')\n    require(tuple(col[i] for i in (3, 7, 11, 15)) == (0, 0, 0, 1), f\'{label} must be affine\')\n    require(core.determinant(transpose(col)) != 0, f\'{label} must be nonsingular\')\n\n\ndef validate_parents(parents):\n    for i, parent in enumerate(parents):\n        require(-1 <= parent < len(parents) and parent != i, \'invalid bone parent\')\n        seen = set()\n        at_index = i\n        while at_index != -1:\n            require(at_index not in seen, \'cycle in bone hierarchy\')\n            seen.add(at_index)\n            at_index = parents[at_index]\n\n\nclass Converter(core.Converter):\n    def __init__(self, doc, blob, *, base_color_only=False):\n        require(not doc.get(\'animations\'), \'embedded animations are not supported; author runtime animation separately\')\n        skins = objects(doc.get(\'skins\', []), \'skins\')\n        require(len(skins) == 1, \'exactly one skin is required\')\n        self.skin = skins[0]\n        self.warnings = []\n        sanitized = copy.deepcopy(doc)\n        sanitized.pop(\'skins\', None)\n        for node in objects(sanitized.get(\'nodes\', []), \'nodes\'):\n            node.pop(\'skin\', None)\n        omitted = []\n        for i, material in enumerate(objects(sanitized.get(\'materials\', []), \'materials\')):\n            pbr = material.get(\'pbrMetallicRoughness\', {})\n            require(isinstance(pbr, dict), \'pbrMetallicRoughness must be an object\')\n            for field in (\'normalTexture\', \'occlusionTexture\'):\n                if field in material:\n                    require(base_color_only, f\'{field} omitted only with explicit --base-color-only\')\n                    material.pop(field)\n                    omitted.append(f\'material {i} {field}\')\n            if \'metallicRoughnessTexture\' in pbr:\n                require(base_color_only, \'metallicRoughnessTexture omitted only with explicit --base-color-only\')\n                pbr.pop(\'metallicRoughnessTexture\')\n                omitted.append(f\'material {i} metallicRoughnessTexture\')\n        for i, sampler in enumerate(objects(sanitized.get(\'samplers\', []), \'samplers\')):\n            if sampler.get(\'minFilter\') in (9985, 9987):\n                require(base_color_only, \'mipmap sampling simplified only with explicit --base-color-only\')\n                sampler[\'minFilter\'] = 9729\n                omitted.append(f\'sampler {i} mipmap minification (output uses LINEAR)\')\n        if base_color_only:\n            self.warnings.append(\'BASE-COLOR-ONLY: omitted \' + (\', \'.join(omitted) if omitted else \'no additional material maps\'))\n        # The original v1 constructor still rejects skins. A private sanitized\n        # validation view merely lets us reuse its strict geometry/image helpers.\n        super().__init__(sanitized, blob)\n        core.reject_extensions(doc)  # Includes extensions on the skin itself.\n        self.doc = doc\n        self.nodes = objects(doc.get(\'nodes\', []), \'nodes\')\n\n    def skin_accessor(self, index, semantic):\n        acc = at(self.accessors, index, semantic)\n        require(\'sparse\' not in acc, \'sparse accessors are not supported\')\n        comp = acc.get(\'componentType\')\n        norm = acc.get(\'normalized\', False)\n        require(type(norm) is bool, \'accessor normalized must be boolean\')\n        require(type(comp) is int, \'accessor componentType must be an integer\')\n        if semantic == \'inverseBindMatrices\':\n            require(comp == 5126 and acc.get(\'type\') == \'MAT4\' and not norm,\n                    \'inverseBindMatrices requires unnormalized FLOAT MAT4\')\n            ncomp, maximum = 16, MAX_BONES\n        elif semantic.startswith(\'JOINTS_\'):\n            require(comp in (5121, 5123) and acc.get(\'type\') == \'VEC4\' and not norm,\n                    \'JOINTS requires unnormalized UNSIGNED_BYTE/SHORT VEC4\')\n            ncomp, maximum = 4, core.MAX_VERTICES\n        else:\n            require(comp in (5121, 5123, 5126) and acc.get(\'type\') == \'VEC4\', \'WEIGHTS requires FLOAT or normalized unsigned VEC4\')\n            require(norm == (comp != 5126), \'integer weights must be normalized; float weights must not be\')\n            ncomp, maximum = 4, core.MAX_VERTICES\n        fmt, width = {5121: (\'B\', 1), 5123: (\'H\', 2), 5126: (\'f\', 4)}[comp]\n        count = uint(acc.get(\'count\'), semantic + \' count\', maximum)\n        require(count > 0, semantic + \' is empty\')\n        view, base, length = self.view(acc.get(\'bufferView\'))\n        offset = uint(acc.get(\'byteOffset\', 0), \'accessor byteOffset\')\n        require(offset % width == 0 and (base + offset) % width == 0, \'accessor must be component aligned\')\n        stride = uint(view.get(\'byteStride\', width * ncomp), \'byteStride\', 252)\n        require(stride >= width * ncomp, \'accessor stride too short\')\n        if \'byteStride\' in view:\n            require(semantic != \'inverseBindMatrices\' and stride % 4 == 0, \'invalid skin accessor byteStride\')\n        require(offset <= length and (count - 1) * stride + width * ncomp <= length - offset, \'skin accessor exceeds bufferView\')\n        unpack = struct.Struct(\'<\' + fmt * ncomp).unpack_from\n        rows = [unpack(self.blob, base + offset + i * stride) for i in range(count)]\n        if norm:\n            divisor = 255 if comp == 5121 else 65535\n            rows = [tuple(v / divisor for v in row) for row in rows]\n        require(all(math.isfinite(v) for row in rows for v in row), \'non-finite skin accessor\')\n        return rows\n\n    def hierarchy(self):\n        require(self.scenes, \'a default scene is required\')\n        scene = at(self.scenes, self.doc.get(\'scene\', 0), \'scene\')\n        roots = scene.get(\'nodes\', [])\n        require(isinstance(roots, list), \'scene nodes must be an array\')\n        parents, globals_, locals_, active = {}, {}, {}, set()\n        def walk(idx, parent, world, depth):\n            idx = uint(idx, \'node index\')\n            node = at(self.nodes, idx, \'node\')\n            require(depth <= core.MAX_DEPTH, \'node hierarchy exceeds 256 levels\')\n            require(idx not in active, \'cycle in node hierarchy\')\n            require(idx not in parents, \'node has multiple parents or duplicate roots\')\n            active.add(idx)\n            parents[idx] = parent\n            local = core.node_matrix(node)\n            global_ = matmul(world, local)\n            matrix_valid(transpose(local), \'node local matrix\')\n            matrix_valid(transpose(global_), \'node global matrix\')\n            locals_[idx], globals_[idx] = local, global_\n            children = node.get(\'children\', [])\n            require(isinstance(children, list), \'node children must be an array\')\n            for child in children:\n                walk(child, idx, global_, depth + 1)\n            active.remove(idx)\n        for root in roots:\n            walk(root, -1, core.IDENTITY, 0)\n        return parents, globals_, locals_\n\n    def convert(self):\n        parents, globals_, locals_ = self.hierarchy()\n        joints = self.skin.get(\'joints\')\n        require(isinstance(joints, list) and 1 <= len(joints) <= MAX_BONES, \'skin must have 1..128 joints\')\n        require(all(type(i) is int and i in parents for i in joints), \'every joint must be a node in the selected scene\')\n        require(len(set(joints)) == len(joints), \'duplicate skin joint\')\n        if \'skeleton\' in self.skin:\n            require(type(self.skin[\'skeleton\']) is int and self.skin[\'skeleton\'] in parents, \'skin skeleton is not in selected scene\')\n        joint_map = {node: index for index, node in enumerate(joints)}\n        binds = self.skin_accessor(self.skin[\'inverseBindMatrices\'], \'inverseBindMatrices\') if \'inverseBindMatrices\' in self.skin else [core.IDENTITY] * len(joints)\n        require(len(binds) == len(joints), \'inverse bind count must equal joint count\')\n        bones = []\n        for index, node in enumerate(joints):\n            parent, local = parents[node], locals_[node]\n            while parent != -1 and parent not in joint_map:\n                local = matmul(locals_[parent], local)\n                parent = parents[parent]\n            name = self.nodes[node].get(\'name\', f\'joint_{index}\')\n            require(isinstance(name, str) and 1 <= len(name.encode(\'utf-8\')) <= 64 and \'\\x00\' not in name,\n                    \'joint names must be 1..64 UTF-8 bytes without NUL\')\n            local = transpose(local)\n            matrix_valid(local, \'bone rest matrix\')\n            matrix_valid(binds[index], \'inverse bind matrix\')\n            bones.append((name, joint_map.get(parent, -1), local, binds[index]))\n        validate_parents([b[1] for b in bones])\n        output = []\n        for node_idx in parents:\n            node = self.nodes[node_idx]\n            if \'mesh\' not in node:\n                require(\'skin\' not in node, \'skin node requires a mesh\')\n                continue\n            require(type(node.get(\'skin\')) is int and node[\'skin\'] == 0, \'all mesh nodes must reference the one skin\')\n            mesh = at(self.meshes, node[\'mesh\'], \'node mesh\')\n            primitives = objects(mesh.get(\'primitives\', []), \'primitives\')\n            require(primitives and len(output) + len(primitives) <= MAX_MESHES, \'mesh primitive count must be 1..128\')\n            # In glTF skinned WORLD output, meshGlobal cancels its inverse in\n            # joint matrices. Do not bake it into positions or inverse binds.\n            for primitive in primitives:\n                output.append(self.skin_primitive(primitive, len(joints)))\n        require(output, \'selected scene has no skinned meshes\')\n        self.bones, self.output_meshes, self.source_globals = bones, output, globals_\n        return encode_vrs(bones, output)\n\n    def skin_primitive(self, primitive, bone_count):\n        attrs = primitive.get(\'attributes\')\n        require(isinstance(attrs, dict), \'primitive attributes must be an object\')\n        supported = {\'POSITION\', \'NORMAL\', \'TEXCOORD_0\', \'JOINTS_0\', \'WEIGHTS_0\', \'JOINTS_1\', \'WEIGHTS_1\'}\n        require(not set(attrs) - supported, \'unsupported vertex attributes\')\n        require(\'JOINTS_0\' in attrs and \'WEIGHTS_0\' in attrs, \'JOINTS_0 and WEIGHTS_0 are required\')\n        require((\'JOINTS_1\' in attrs) == (\'WEIGHTS_1\' in attrs), \'JOINTS_1 and WEIGHTS_1 must occur together\')\n        static = dict(primitive)\n        static[\'attributes\'] = {k: v for k, v in attrs.items() if k in {\'POSITION\', \'NORMAL\', \'TEXCOORD_0\'}}\n        factors, w, h, raster, vertices, indices = super().primitive(static, core.IDENTITY)\n        joint_rows = self.skin_accessor(attrs[\'JOINTS_0\'], \'JOINTS_0\')\n        weight_rows = self.skin_accessor(attrs[\'WEIGHTS_0\'], \'WEIGHTS_0\')\n        require(len(joint_rows) == len(weight_rows) == len(vertices), \'skin attribute counts must match POSITION\')\n        if \'JOINTS_1\' in attrs:\n            j1 = self.skin_accessor(attrs[\'JOINTS_1\'], \'JOINTS_1\')\n            w1 = self.skin_accessor(attrs[\'WEIGHTS_1\'], \'WEIGHTS_1\')\n            require(len(j1) == len(w1) == len(vertices), \'skin attribute counts must match POSITION\')\n            joint_rows = [a + b for a, b in zip(joint_rows, j1)]\n            weight_rows = [a + b for a, b in zip(weight_rows, w1)]\n        result = []\n        for vertex, joints, weights in zip(vertices, joint_rows, weight_rows):\n            require(all(0 <= j < bone_count for j in joints), \'vertex joint index outside skin\')\n            require(all(0 <= w <= 1 for w in weights) and sum(weights) > 0, \'weights must be nonnegative, <=1 and have positive sum\')\n            total = sum(weights)\n            js = tuple(joints) + (0,) * (8 - len(joints))\n            ws = tuple(w / total for w in weights) + (0.0,) * (8 - len(weights))\n            result.append(vertex + js + ws)\n        return factors, w, h, raster, result, indices\n\n\ndef encode_vrs(bones, meshes):\n    payload = bytearray(struct.pack(\'<I\', len(bones)))\n    for name, parent, rest, inverse in bones:\n        name = name.encode(\'utf-8\')\n        payload.extend(struct.pack(\'<H\', len(name)))\n        payload.extend(name)\n        payload.extend(struct.pack(\'<i32f\', parent, *rest, *inverse))\n    payload.extend(struct.pack(\'<I\', len(meshes)))\n    for factors, width, height, raster, vertices, indices in meshes:\n        payload.extend(struct.pack(\'<6fIII\', *factors, width, height, len(raster)))\n        payload.extend(raster)\n        payload.extend(struct.pack(\'<II\', len(vertices), len(indices)))\n        for vertex in vertices:\n            payload.extend(VERTEX.pack(*vertex))\n        for start in range(0, len(indices), 16384):\n            part = indices[start:start + 16384]\n            payload.extend(struct.pack(\'<\' + \'I\' * len(part), *part))\n        require(len(payload) + HEADER.size <= core.MAX_FILE, \'output exceeds 128 MiB\')\n    data = HEADER.pack(MAGIC, VERSION, len(payload), zlib.crc32(payload), 0) + payload\n    inspect_vrs(data)\n    return data\n\n\ndef inspect_vrs(data):\n    require(HEADER.size <= len(data) <= core.MAX_FILE, \'VRS is truncated or exceeds 128 MiB\')\n    magic, version, length, checksum, reserved = HEADER.unpack_from(data)\n    require(magic == MAGIC and version == VERSION, \'invalid VRS magic/version\')\n    require(reserved == 0 and length == len(data) - HEADER.size, \'invalid VRS header\')\n    payload = memoryview(data)[HEADER.size:]\n    require(zlib.crc32(payload) == checksum, \'VRS checksum mismatch\')\n    cursor = 0\n    def take(n):\n        nonlocal cursor\n        require(n <= len(payload) - cursor, \'truncated VRS payload\')\n        result = payload[cursor:cursor + n]\n        cursor += n\n        return result\n    def unpack(fmt):\n        return struct.unpack(fmt, take(struct.calcsize(fmt)))\n    count, = unpack(\'<I\')\n    require(1 <= count <= MAX_BONES, \'bone count outside 1..128\')\n    parents = []\n    for _ in range(count):\n        size, = unpack(\'<H\')\n        require(1 <= size <= 64, \'invalid bone name length\')\n        try:\n            name = bytes(take(size)).decode(\'utf-8\')\n        except UnicodeDecodeError as exc:\n            raise AssetError(\'invalid UTF-8 bone name\') from exc\n        require(\'\\x00\' not in name, \'NUL in bone name\')\n        parent, = unpack(\'<i\')\n        parents.append(parent)\n        matrix_valid(unpack(\'<16f\'), \'bone rest matrix\')\n        matrix_valid(unpack(\'<16f\'), \'inverse bind matrix\')\n    validate_parents(parents)\n    mesh_count, = unpack(\'<I\')\n    require(1 <= mesh_count <= MAX_MESHES, \'mesh count outside 1..128\')\n    total_v = total_i = total_t = textures = 0\n    bounds_min, bounds_max = [math.inf] * 3, [-math.inf] * 3\n    for _ in range(mesh_count):\n        factors = unpack(\'<6f\')\n        require(all(math.isfinite(x) and 0 <= x <= 1 for x in factors), \'invalid material factor\')\n        width, height, size = unpack(\'<III\')\n        require(width <= core.MAX_TEXTURE_DIM and height <= core.MAX_TEXTURE_DIM, \'texture too large\')\n        require((width == height == size == 0) or (width > 0 and height > 0 and size == width * height * 4), \'invalid RGBA dimensions\')\n        total_t += size\n        require(total_t <= core.MAX_TEXTURE_BYTES, \'aggregate texture budget exceeded\')\n        take(size)\n        textures += int(size > 0)\n        vc, ic = unpack(\'<II\')\n        require(vc > 0 and ic > 0 and ic % 3 == 0, \'invalid triangle geometry\')\n        total_v += vc\n        total_i += ic\n        require(total_v <= core.MAX_VERTICES and total_i <= core.MAX_INDICES, \'aggregate geometry budget exceeded\')\n        require(vc * VERTEX.size + ic * 4 <= len(payload) - cursor, \'truncated geometry\')\n        for _ in range(vc):\n            v = unpack(\'<8f8H8f\')\n            require(all(math.isfinite(x) for x in v), \'non-finite vertex\')\n            require(all(abs(x) <= core.MAX_POSITION for x in v[:3]) and all(abs(x) <= core.MAX_UV for x in v[6:8]), \'vertex coordinates exceed bounds\')\n            require(all(abs(x) <= 1.01 for x in v[3:6]), \'normal must be normalized\')\n            n2 = 0.0\n            for x in v[3:6]:\n                n2 = core.f32(n2 + core.f32(x * x))\n            require(core.f32(.98) <= n2 <= core.f32(1.02), \'normal must be normalized\')\n            require(all(0 <= j < count for j in v[8:16]), \'joint index outside skin\')\n            weights = v[16:24]\n            require(all(0 <= x <= 1 for x in weights), \'weights must be nonnegative and <=1\')\n            s = 0.0\n            for x in weights:\n                s = core.f32(s + x)\n            require(all(0 <= x <= 1 for x in weights) and core.f32(.9999) <= s <= core.f32(1.0001), \'weights must be normalized\')\n            for j in range(3):\n                bounds_min[j], bounds_max[j] = min(bounds_min[j], v[j]), max(bounds_max[j], v[j])\n        for _ in range(ic):\n            idx, = unpack(\'<I\')\n            require(idx < vc, \'index outside vertex array\')\n    require(cursor == len(payload), \'trailing VRS payload bytes\')\n    return {\'format\': MAGIC.decode(), \'version\': version, \'bones\': count, \'meshes\': mesh_count,\n            \'vertices\': total_v, \'indices\': total_i, \'triangles\': total_i // 3,\n            \'textures\': textures, \'texture_bytes\': total_t, \'file_bytes\': len(data),\n            \'crc32\': f\'{checksum:08x}\', \'bounds_min\': bounds_min, \'bounds_max\': bounds_max}\n\n\ndef convert_glb(data, *, base_color_only=False):\n    doc, blob = core.parse_glb(data)\n    converter = Converter(doc, blob, base_color_only=base_color_only)\n    result = converter.convert()\n    return result, converter\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    pack = commands.add_parser(\'pack\')\n    pack.add_argument(\'input\', type=Path)\n    pack.add_argument(\'output\', type=Path)\n    pack.add_argument(\'--base-color-only\', action=\'store_true\', help=\'explicitly omit normal/ORM textures and simplify linear mipmap sampling\')\n    pack.add_argument(\'--force\', action=\'store_true\')\n    inspect = commands.add_parser(\'inspect\')\n    inspect.add_argument(\'input\', type=Path)\n    args = parser.parse_args(argv)\n    try:\n        source = core.read_limited(args.input)\n        if args.command == \'pack\':\n            require(args.input.resolve() != args.output.resolve(), \'input and output paths must differ\')\n            result, converter = convert_glb(source, base_color_only=args.base_color_only)\n            for warning in converter.warnings:\n                print(warning, file=sys.stderr)\n            core.atomic_write(args.output, result, args.force)\n            print(json.dumps(inspect_vrs(result), indent=2, sort_keys=True))\n        else:\n            print(json.dumps(inspect_vrs(source), indent=2, sort_keys=True))\n        return 0\n    except (AssetError, OSError, OverflowError, RecursionError, struct.error, UnicodeEncodeError) as exc:\n        print(f\'vrskin: {exc}\', file=sys.stderr)\n        return 1\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n\n', 'vrview': '#!/usr/bin/env python3\n"""Compile evaluated Blender glTF clips to VRSKIN02 + VRMESH01 + VRANIM01.\n\nGeometry helpers stay unchanged. All tracks are glTF Y-up; the runtime alone\napplies Ry(pi). Animation is baked source data, never runtime IK or pose edits.\n"""\nfrom __future__ import annotations\nimport argparse\nimport bisect\nimport copy\nimport json\nimport math\nfrom pathlib import Path\nimport struct\nimport sys\nimport zlib\nimport vrpack as core\nimport vrskin\nfrom vrpack import AssetError, require, uint, number, objects, at, matmul\n\nMAGIC = b\'VRANIM01\'\nVERSION = 1\nHEADER = struct.Struct(\'<8sIIII\')\nTRANSFORM = struct.Struct(\'<10f\')\nMAX_TRANSFORMS = 2_000_000\nSCHEMA = \'rust-duty-viewmodel-export/v1\'\n\n\ndef string(value, label=\'name\', maximum=128):\n    require(isinstance(value, str) and 1 <= len(value.encode(\'utf-8\')) <= maximum and \'\\0\' not in value,\n            f\'{label} must be 1..{maximum} UTF-8 bytes without NUL\')\n    return value\n\n\ndef inverse(m):\n    """Invert a row-major affine matrix, rejecting singular matrices."""\n    vrskin.matrix_valid(vrskin.transpose(m), \'affine matrix\')\n    n = core.normal_matrix(m)\n    r = (n[0], n[3], n[6], 0, n[1], n[4], n[7], 0, n[2], n[5], n[8], 0, 0, 0, 0, 1)\n    r = list(r)\n    for i in range(3):\n        r[4*i+3] = -sum(r[4*i+j] * m[4*j+3] for j in range(3))\n    return tuple(r)\n\n\ndef validate_transform(trs):\n    require(len(trs) == 10 and all(math.isfinite(v) for v in trs), \'non-finite or invalid TRS\')\n    require(all(abs(v) <= 10000 for v in trs[:3]), \'translation exceeds +/-10000\')\n    require(abs(math.hypot(*trs[3:7])-1) <= 1e-4, \'quaternion must be normalized within 1e-4\')\n    require(all(1e-8 < v <= 10000 for v in trs[7:]), \'scale must be positive, >1e-8 and <=10000\')\n    return tuple(trs[:3]) + core.normalized(trs[3:7], \'quaternion\') + tuple(trs[7:])\n\n\ndef decompose(m):\n    """TRS only: reject reflected/singular/sheared matrices instead of losing data."""\n    vrskin.matrix_valid(vrskin.transpose(m), \'animated matrix\')\n    t = (m[3], m[7], m[11])\n    s = tuple(math.hypot(*(m[r*4+c] for r in range(3))) for c in range(3))\n    require(all(v > 1e-8 for v in s) and core.determinant(m) > 0, \'negative or singular animated scale\')\n    r = tuple(m[row*4+c]/s[c] for row in range(3) for c in range(3))\n    require(all(abs(sum(r[k*3+i]*r[k*3+j] for k in range(3)) - (i == j)) <= 2e-5\n                for i in range(3) for j in range(3)), \'animated transform contains shear\')\n    trace = r[0]+r[4]+r[8]\n    if trace > 0:\n        h = 2*math.sqrt(trace+1)\n        q = ((r[7]-r[5])/h, (r[2]-r[6])/h, (r[3]-r[1])/h, h/4)\n    else:\n        i = max(range(3), key=lambda k: r[k*3+k]); j=(i+1)%3; k=(i+2)%3\n        h = 2*math.sqrt(max(0, 1+r[i*3+i]-r[j*3+j]-r[k*3+k]))\n        require(h > 0, \'invalid rotation matrix\')\n        v = [0.0]*4; v[i]=h/4; v[j]=(r[i*3+j]+r[j*3+i])/h\n        v[k]=(r[i*3+k]+r[k*3+i])/h; v[3]=(r[k*3+j]-r[j*3+k])/h\n        q = tuple(v)\n    return validate_transform(t + core.normalized(q, \'rotation\') + s)\n\n\ndef matrix(trs):\n    return core.node_matrix({\'translation\': list(trs[:3]), \'rotation\': list(trs[3:7]), \'scale\': list(trs[7:])})\n\n\ndef slerp(a, b, factor):\n    dot = sum(x*y for x,y in zip(a,b))\n    if dot < 0: b=tuple(-v for v in b); dot=-dot\n    dot = max(-1, min(1, dot))\n    if dot > .9995:\n        return core.normalized(tuple(x+(y-x)*factor for x,y in zip(a,b)), \'interpolated quaternion\')\n    theta=math.acos(dot); divisor=math.sin(theta)\n    return tuple((math.sin((1-factor)*theta)*x+math.sin(factor*theta)*y)/divisor for x,y in zip(a,b))\n\n\ndef interpolate(a, b, factor):\n    return tuple(x+(y-x)*factor for x,y in zip(a[:3],b[:3])) + slerp(a[3:7],b[3:7],factor) + tuple(x+(y-x)*factor for x,y in zip(a[7:],b[7:]))\n\n\nclass Scene:\n    """A strict sampled glTF scene. Original node indices remain unchanged."""\n    def __init__(self, doc, blob, *, base_color_only=False, single_sided_materials=False, armature=None, rigid_armatures=()):\n        self.doc, self.blob = doc, blob\n        core.reject_extensions(doc)\n        source_nodes=objects(doc.get(\'nodes\',[]),\'nodes\')\n        skins=objects(doc.get(\'skins\',[]),\'skins\'); require(skins,\'at least one skin is required\')\n        # Ownership follows actual glTF ancestry, never hardcoded rig/bone names.\n        direct_parents={}\n        for i,node in enumerate(source_nodes):\n            children=node.get(\'children\',[]); require(isinstance(children,list),\'node children must be an array\')\n            for child in children:\n                at(source_nodes,child,\'child node\'); require(child not in direct_parents,\'node has multiple parents\')\n                direct_parents[child]=i\n        def belongs(skin_index,owner):\n            matches=[i for i,n in enumerate(source_nodes) if n.get(\'name\')==owner]\n            require(len(matches)==1,f\'armature name must be unique in glTF: {owner}\')\n            joints=skins[skin_index].get(\'joints\')\n            require(isinstance(joints,list) and joints,\'skin has no joints\')\n            for joint in joints:\n                at(source_nodes,joint,\'skin joint\'); current=joint; seen=set()\n                while current!=matches[0] and current in direct_parents:\n                    require(current not in seen,\'cycle in skin ancestry\'); seen.add(current); current=direct_parents[current]\n                if current!=matches[0]: return False\n            return True\n        named=[i for i,s in enumerate(skins) if s.get(\'name\')==armature] if armature else []\n        primary=named or ([i for i in range(len(skins)) if belongs(i,armature)] if armature else list(range(len(skins))))\n        require(len(primary)==1,\'deform armature must identify exactly one skin\')\n        self.primary_skin=primary[0]; self.rigid_skins={}\n        require(isinstance(rigid_armatures,(list,tuple)) and all(isinstance(n,str) for n in rigid_armatures) and len(set(rigid_armatures))==len(rigid_armatures),\'invalid rigid_armatures\')\n        for i,skin in enumerate(skins):\n            if i==self.primary_skin: continue\n            if not any(n.get(\'skin\')==i for n in source_nodes): continue\n            require(any(belongs(i,owner) for owner in rigid_armatures),\'extra skin requires explicit rigid_armatures authorization\')\n            require(isinstance(skin.get(\'joints\'),list) and 1<=len(skin[\'joints\'])<=128,\'rigid skin joint count outside 1..128\')\n            self.rigid_skins[i]=skin\n        sanitized = copy.deepcopy(doc)\n        self.warnings=[]\n        if single_sided_materials:\n            for index,material in enumerate(objects(sanitized.get(\'materials\',[]),\'materials\')):\n                if material.get(\'doubleSided\') is True:\n                    material[\'doubleSided\']=False\n                    self.warnings.append(f\'SINGLE-SIDED-MATERIALS: material {index} {material.get("name", "<unnamed>")}: doubleSided omitted\')\n        sanitized.pop(\'animations\', None); sanitized[\'skins\']=[copy.deepcopy(skins[self.primary_skin])]\n        for node in objects(sanitized.get(\'nodes\', []), \'nodes\'):\n            if \'skin\' in node:\n                at(skins,node[\'skin\'],\'node skin\')\n                if node[\'skin\']==self.primary_skin: node[\'skin\']=0\n                else: node.pop(\'mesh\',None); node.pop(\'skin\')\n            else: node.pop(\'mesh\',None)\n        self.skin = vrskin.Converter(sanitized, blob, base_color_only=base_color_only)\n        self.vrs = self.skin.convert()\n        self.parents, self.rest_globals, self.rest_locals = self.skin.hierarchy()\n        self.nodes = objects(doc.get(\'nodes\', []), \'nodes\')\n        self.joints = self.skin.skin[\'joints\']\n        self.joint_map = {n:i for i,n in enumerate(self.joints)}\n        self.names = {}\n        for idx in self.parents:\n            name = self.nodes[idx].get(\'name\')\n            if name is not None:\n                require(isinstance(name,str), \'node name must be a string\')\n                self.names.setdefault(name, []).append(idx)\n        # A separate validation view retains sanitized materials from the skin\n        # helper and removes skin attributes only when compiling rigid meshes.\n        geom = copy.deepcopy(self.skin.doc)\n        geom.pop(\'skins\', None)\n        for n in geom[\'nodes\']: n.pop(\'skin\', None)\n        self.geometry = core.Converter(self._materials(geom, base_color_only), blob)\n        self.rigid_bindings={}\n        for skin_index,skin in self.rigid_skins.items():\n            joints=skin[\'joints\']; weighted=set(); nodes=[]\n            require(len(set(joints))==len(joints),\'duplicate rigid skin joints\')\n            require(all(type(j) is int and j in self.parents for j in joints),\'rigid skin joints must be in selected scene\')\n            binds=self.skin.skin_accessor(skin[\'inverseBindMatrices\'],\'inverseBindMatrices\') if \'inverseBindMatrices\' in skin else [core.IDENTITY]*len(joints)\n            require(len(binds)==len(joints),\'rigid inverse bind count mismatch\')\n            for bind in binds: vrskin.matrix_valid(bind,\'rigid inverse bind\')\n            for idx,node in enumerate(self.nodes):\n                if node.get(\'skin\')!=skin_index: continue\n                require(idx in self.parents,\'rigid skin must be in selected scene\'); nodes.append(idx)\n                mesh=at(self.skin.meshes,node.get(\'mesh\'),\'rigid skin mesh\')\n                for primitive in objects(mesh.get(\'primitives\',[]),\'rigid primitives\'):\n                    # The unchanged skin validator checks all eight indices,\n                    # vertex counts and finite weights before rigid conversion.\n                    self.skin.skin_primitive(primitive,len(joints))\n                    attrs=primitive[\'attributes\']\n                    js=self.skin.skin_accessor(attrs[\'JOINTS_0\'],\'JOINTS_0\')\n                    ws=self.skin.skin_accessor(attrs[\'WEIGHTS_0\'],\'WEIGHTS_0\')\n                    if \'JOINTS_1\' in attrs:\n                        js=[a+b for a,b in zip(js,self.skin.skin_accessor(attrs[\'JOINTS_1\'],\'JOINTS_1\'))]\n                        ws=[a+b for a,b in zip(ws,self.skin.skin_accessor(attrs[\'WEIGHTS_1\'],\'WEIGHTS_1\'))]\n                    require(all(abs(sum(row)-1)<=1e-4 for row in ws),\'rigid skin weights must sum to one\')\n                    weighted.update(j for jr,wr in zip(js,ws) for j,w in zip(jr,wr) if w>0)\n            require(len(weighted)==1,\'rigid skin must have exactly one positively weighted joint across all meshes\')\n            joint_index=next(iter(weighted))\n            for idx in nodes: self.rigid_bindings[idx]=(joints[joint_index],vrskin.transpose(binds[joint_index]))\n        self.channels = {}\n        self.parse_animation()\n\n    @staticmethod\n    def _materials(doc, base_color_only):\n        if base_color_only:\n            for m in doc.get(\'materials\', []):\n                m.pop(\'normalTexture\',None); m.pop(\'occlusionTexture\',None)\n                m.get(\'pbrMetallicRoughness\',{}).pop(\'metallicRoughnessTexture\',None)\n            for s in doc.get(\'samplers\',[]):\n                if s.get(\'minFilter\') in (9985,9987): s[\'minFilter\']=9729\n        return doc\n\n    def node_named(self, name):\n        string(name, \'actor name\')\n        matches = self.names.get(name, [])\n        require(len(matches) == 1, f\'node name must identify exactly one exported node: {name}\')\n        return matches[0]\n\n    def accessor(self, index, kind):\n        acc = at(self.skin.accessors, index, \'animation accessor\')\n        require(acc.get(\'componentType\') == 5126 and acc.get(\'type\') == kind and acc.get(\'normalized\',False) is False,\n                f\'animation accessor requires unnormalized FLOAT {kind}\')\n        require(\'sparse\' not in acc, \'sparse animation accessors are unsupported\')\n        n = {\'SCALAR\':1,\'VEC3\':3,\'VEC4\':4}[kind]\n        count = uint(acc.get(\'count\'),\'animation accessor count\',65536)\n        require(count > 0, \'empty animation accessor\')\n        view, base, length = self.skin.view(acc.get(\'bufferView\'))\n        require(\'byteStride\' not in view, \'animation accessors must be tightly packed\')\n        offset = uint(acc.get(\'byteOffset\',0),\'animation byte offset\')\n        require((base+offset)%4 == 0 and offset%4 == 0, \'unaligned animation accessor\')\n        require(offset <= length and count*n*4 <= length-offset, \'animation accessor exceeds bufferView\')\n        rows = [struct.unpack_from(\'<\'+\'f\'*n,self.blob,base+offset+i*n*4) for i in range(count)]\n        require(all(math.isfinite(x) for row in rows for x in row), \'non-finite animation accessor\')\n        return rows\n\n    def parse_animation(self):\n        animations = objects(self.doc.get(\'animations\',[]),\'animations\')\n        require(len(animations) <= 1, \'each GLB must contain one independently baked SCENE animation\')\n        if not animations: return\n        samplers = objects(animations[0].get(\'samplers\',[]),\'animation samplers\')\n        channels = objects(animations[0].get(\'channels\',[]),\'animation channels\')\n        require(channels and len(channels) <= len(self.parents)*3, \'empty or excessive animation channels\')\n        used=set()\n        for channel in channels:\n            target=channel.get(\'target\'); require(isinstance(target,dict),\'animation target must be an object\')\n            idx=uint(target.get(\'node\'),\'animation target node\'); path=target.get(\'path\')\n            require(idx in self.parents, \'animation targets node outside selected scene\')\n            require(path in (\'translation\',\'rotation\',\'scale\'),\'only evaluated TRS animation is supported\')\n            require(\'matrix\' not in self.nodes[idx], \'animated nodes must use TRS, not matrix\')\n            require((idx,path) not in self.channels, \'duplicate animation channel\')\n            sampler_index=uint(channel.get(\'sampler\'),\'animation sampler\')\n            sampler=at(samplers,sampler_index,\'animation sampler\'); used.add(sampler_index)\n            mode=sampler.get(\'interpolation\',\'LINEAR\')\n            require(mode in (\'LINEAR\',\'STEP\'), \'animation must be baked LINEAR/STEP, not cubic\')\n            times=[r[0] for r in self.accessor(sampler.get(\'input\'),\'SCALAR\')]\n            require(times[0] >= 0 and all(b>a for a,b in zip(times,times[1:])), \'animation times must be nonnegative and strictly increasing\')\n            values=self.accessor(sampler.get(\'output\'),\'VEC4\' if path==\'rotation\' else \'VEC3\')\n            require(len(times)==len(values),\'animation input/output count mismatch\')\n            if path==\'rotation\':\n                require(all(abs(math.hypot(*v)-1)<=1e-4 for v in values),\'animation quaternion must be normalized\')\n                values=[core.normalized(v,\'animation rotation\') for v in values]\n            elif path==\'scale\': require(all(1e-8<x<=10000 for v in values for x in v),\'invalid animation scale\')\n            else: require(all(abs(x)<=10000 for v in values for x in v),\'animation translation exceeds bounds\')\n            self.channels[idx,path]=(times,values,mode)\n        require(used==set(range(len(samplers))), \'unreferenced animation sampler\')\n\n    def sample(self, time):\n        locals_={}\n        for idx in self.parents:\n            node=self.nodes[idx]\n            if not any((idx,p) in self.channels for p in (\'translation\',\'rotation\',\'scale\')):\n                locals_[idx]=self.rest_locals[idx]; continue\n            values={p:node.get(p,d) for p,d in [(\'translation\',[0,0,0]),(\'rotation\',[0,0,0,1]),(\'scale\',[1,1,1])]}\n            for path in values:\n                if (idx,path) not in self.channels: continue\n                times,rows,mode=self.channels[idx,path]\n                i=max(0,bisect.bisect_right(times,time)-1)\n                if i==len(times)-1 or time<=times[0] or mode==\'STEP\': value=rows[i]\n                else:\n                    f=(time-times[i])/(times[i+1]-times[i])\n                    value=slerp(rows[i],rows[i+1],f) if path==\'rotation\' else tuple(a+(b-a)*f for a,b in zip(rows[i],rows[i+1]))\n                values[path]=list(value)\n            locals_[idx]=core.node_matrix(values)\n        globals_={}\n        for idx,parent in self.parents.items():\n            globals_[idx]=matmul(globals_[parent],locals_[idx]) if parent!=-1 else locals_[idx]\n        bones=[]\n        for idx in self.joints:\n            local=locals_[idx]; parent=self.parents[idx]\n            while parent!=-1 and parent not in self.joint_map:\n                local=matmul(locals_[parent],local); parent=self.parents[parent]\n            bones.append(decompose(local))\n        return bones, globals_\n\n    def rigid_signature(self, name):\n        """Compare local geometry/materials, separate from animated placement."""\n        idx=self.node_named(name); node=self.nodes[idx]\n        binding=self.rigid_bindings.get(idx)\n        result=[]\n        if \'mesh\' in node:\n            validator=core.Converter(self.geometry.doc,self.blob)\n            mesh=at(validator.meshes,node[\'mesh\'],\'actor mesh\')\n            for primitive in objects(mesh.get(\'primitives\',[]),\'actor primitives\'):\n                if binding:\n                    primitive=dict(primitive)\n                    primitive[\'attributes\']={k:v for k,v in primitive[\'attributes\'].items() if k in (\'POSITION\',\'NORMAL\',\'TEXCOORD_0\')}\n                result.append(validator.primitive(primitive,core.IDENTITY))\n        return result, (self.nodes[binding[0]].get(\'name\'),binding[1]) if binding else None\n\n    def actor_node(self, idx):\n        return self.rigid_bindings[idx][0] if idx in self.rigid_bindings else idx\n\n    def companions(self, actor_names):\n        actors=[]; meshes=[]; assigned=set()\n        for name in actor_names:\n            idx=self.node_named(name); node=self.nodes[idx]\n            require(idx not in self.joint_map and (\'skin\' not in node or idx in self.rigid_bindings),\'actors cannot be deform joints or deform-skinned meshes\')\n            require(idx not in assigned,\'duplicate actor node\'); assigned.add(idx)\n            bound=[]\n            if \'mesh\' in node:\n                mesh=at(self.geometry.meshes,node[\'mesh\'],\'actor mesh\')\n                primitives=objects(mesh.get(\'primitives\',[]),\'actor primitives\')\n                require(primitives, \'actor mesh has no primitives\')\n                for primitive in primitives:\n                    require(len(meshes)<core.MAX_MESHES,\'too many rigid primitives\')\n                    bound.append(len(meshes))\n                    world=self.rest_globals[idx]\n                    if idx in self.rigid_bindings:\n                        joint,bind=self.rigid_bindings[idx]\n                        attrs=primitive.get(\'attributes\',{})\n                        world=matmul(self.rest_globals[joint],bind)\n                        primitive=dict(primitive)\n                        primitive[\'attributes\']={k:v for k,v in attrs.items() if k in (\'POSITION\',\'NORMAL\',\'TEXCOORD_0\')}\n                    meshes.append(self.geometry.primitive(primitive,world))\n            animated_idx=self.actor_node(idx)\n            actors.append({\'name\':name,\'meshes\':bound,\'inverse_rest_global\':vrskin.transpose(inverse(self.rest_globals[animated_idx]))})\n        require(all(i in assigned for i in self.parents if \'mesh\' in self.nodes[i] and (\'skin\' not in self.nodes[i] or i in self.rigid_bindings)),\n                \'every rigid mesh node must be assigned to an actor\')\n        require(meshes, \'VRMESH01 requires at least one rigid mesh\')\n        return self.vrs,core.encode_vrm(meshes),actors\n\n\ndef encode_vra(bones, actors, clips, vrs, vrm):\n    require(1<=len(bones)<=128 and len(actors)<=256 and 1<=len(clips)<=128,\'invalid animation set counts\')\n    total=0\n    for clip in clips:\n        frames=clip.get(\'frames\',[]); require(1<=len(frames)<=65536,\'invalid frame count\')\n        total+=len(frames)*(len(bones)+len(actors)); require(total<=MAX_TRANSFORMS,\'aggregate transform budget exceeded\')\n        require(type(clip.get(\'loop\')) is bool,\'clip loop must be boolean\')\n        for frame in frames:\n            require(len(frame[\'bones\'])==len(bones) and len(frame[\'actors\'])==len(actors) and len(frame[\'visible\'])==len(actors),\'frame channel count mismatch\')\n            require(all(type(v) is int and v in (0,1) for v in frame[\'visible\']),\'invalid visibility\')\n    payload=bytearray(struct.pack(\'<III\',zlib.crc32(vrs),zlib.crc32(vrm),len(bones)))\n    def name(s):\n        raw=string(s).encode(); payload.extend(struct.pack(\'<H\',len(raw))); payload.extend(raw)\n    for b in bones: name(b[0]); payload.extend(struct.pack(\'<i\',b[1]))\n    payload.extend(struct.pack(\'<I\',len(actors)))\n    for actor in actors:\n        name(actor[\'name\']); meshes=actor[\'meshes\']\n        payload.extend(struct.pack(\'<I\',len(meshes)))\n        for i in meshes: payload.extend(struct.pack(\'<I\',i))\n        payload.extend(struct.pack(\'<16f\',*actor[\'inverse_rest_global\']))\n    payload.extend(struct.pack(\'<I\',len(clips)))\n    for clip in clips:\n        name(clip[\'name\']); frames=clip[\'frames\']\n        payload.extend(struct.pack(\'<II\',int(clip[\'loop\']),len(frames)))\n        for frame in frames:\n            payload.extend(struct.pack(\'<f\',frame[\'time\']))\n            for value in frame[\'bones\']+frame[\'actors\']: payload.extend(TRANSFORM.pack(*validate_transform(value)))\n            payload.extend(bytes(frame[\'visible\']))\n            require(len(payload)+HEADER.size<=core.MAX_FILE,\'VRA exceeds 128 MiB\')\n    result=HEADER.pack(MAGIC,VERSION,len(payload),zlib.crc32(payload),0)+payload\n    decode_vra(result,vrs=vrs,vrm=vrm)\n    return result\n\n\ndef decode_vra(data, *, vrs=None, vrm=None):\n    require(HEADER.size<=len(data)<=core.MAX_FILE,\'VRA truncated or exceeds 128 MiB\')\n    magic,version,length,checksum,reserved=HEADER.unpack_from(data)\n    require(magic==MAGIC and version==VERSION,\'invalid VRA magic/version\')\n    require(reserved==0 and length==len(data)-HEADER.size,\'invalid VRA header\')\n    payload=memoryview(data)[HEADER.size:]; require(zlib.crc32(payload)==checksum,\'VRA checksum mismatch\')\n    cursor=0\n    def take(n):\n        nonlocal cursor\n        require(0<=n<=len(payload)-cursor,\'truncated VRA payload\')\n        result=payload[cursor:cursor+n]; cursor+=n; return result\n    def unpack(fmt): return struct.unpack(fmt,take(struct.calcsize(fmt)))\n    def name():\n        n,=unpack(\'<H\'); require(1<=n<=128,\'invalid VRA string length\')\n        try: return string(bytes(take(n)).decode(\'utf-8\'))\n        except UnicodeDecodeError as exc: raise AssetError(\'invalid VRA UTF-8\') from exc\n    skin_crc,mesh_crc,bc=unpack(\'<III\'); require(1<=bc<=128,\'bone count outside 1..128\')\n    bones=[(name(),unpack(\'<i\')[0]) for _ in range(bc)]; vrskin.validate_parents([b[1] for b in bones])\n    ac,=unpack(\'<I\'); require(ac<=256,\'actor count exceeds 256\')\n    actors=[]; actor_names=set(); assigned=set()\n    for _ in range(ac):\n        n=name(); require(n not in actor_names,\'duplicate actor name\'); actor_names.add(n)\n        count,=unpack(\'<I\'); require(count<=256,\'too many actor meshes\'); meshes=[]\n        for _ in range(count):\n            i,=unpack(\'<I\'); require(i<256 and i not in assigned,\'invalid or duplicate actor mesh assignment\')\n            assigned.add(i); meshes.append(i)\n        inv=unpack(\'<16f\'); vrskin.matrix_valid(inv,\'actor inverse rest global\')\n        actors.append({\'name\':n,\'meshes\':meshes,\'inverse_rest_global\':inv})\n    cc,=unpack(\'<I\'); require(1<=cc<=128,\'clip count outside 1..128\')\n    clips=[]; names=set(); total=0\n    for _ in range(cc):\n        n=name(); require(n not in names,\'duplicate clip name\'); names.add(n)\n        flags,fc=unpack(\'<II\'); require(flags<=1 and 1<=fc<=65536,\'invalid clip flags/frame count\')\n        total+=fc*(bc+ac); require(total<=MAX_TRANSFORMS,\'aggregate transform budget exceeded\')\n        require(fc*(4+(bc+ac)*40+ac)<=len(payload)-cursor,\'truncated animation frames\')\n        frames=[]; previous=-1\n        for i in range(fc):\n            t,=unpack(\'<f\'); require(math.isfinite(t) and (t==0 if i==0 else t>previous),\'invalid frame times\'); previous=t\n            values=[validate_transform(unpack(\'<10f\')) for _ in range(bc+ac)]\n            visible=list(take(ac)); require(all(v in (0,1) for v in visible),\'invalid visibility\')\n            frames.append({\'time\':t,\'bones\':values[:bc],\'actors\':values[bc:],\'visible\':visible})\n        clips.append({\'name\':n,\'loop\':bool(flags),\'frames\':frames})\n    require(cursor==len(payload),\'trailing VRA payload bytes\')\n    if vrs is not None:\n        vrskin.inspect_vrs(vrs); require(zlib.crc32(vrs)==skin_crc,\'VRS companion checksum mismatch\')\n        p=28; actual=[]; count=struct.unpack_from(\'<I\',vrs,24)[0]\n        for _ in range(count):\n            size=struct.unpack_from(\'<H\',vrs,p)[0]; p+=2\n            n=vrs[p:p+size].decode(); p+=size; parent=struct.unpack_from(\'<i\',vrs,p)[0]; p+=132; actual.append((n,parent))\n        require(actual==bones,\'VRS companion skeleton mismatch\')\n    if vrm is not None:\n        info=core.inspect_vrm(vrm); require(zlib.crc32(vrm)==mesh_crc,\'VRM companion checksum mismatch\')\n        require(assigned==set(range(info[\'meshes\'])),\'rigid companion meshes must be assigned exactly once\')\n    return {\'bones\':bones,\'actors\':actors,\'clips\':clips,\'skin_crc\':skin_crc,\'mesh_crc\':mesh_crc}\n\n\ndef sample(pack, clip_name, time, *, clamp=False):\n    clips=[c for c in pack[\'clips\'] if c[\'name\']==clip_name]; require(len(clips)==1,\'unknown clip\')\n    clip=clips[0]; frames=clip[\'frames\']; duration=frames[-1][\'time\']; time=core.f32(number(time,\'sample time\'))\n    time=(time%duration) if clip[\'loop\'] and duration>0 and not clamp else max(0,min(time,duration))\n    i=max(0,bisect.bisect_right([f[\'time\'] for f in frames],time)-1); a=frames[i]\n    if i==len(frames)-1: return copy.deepcopy(a)\n    b=frames[i+1]; t=(time-a[\'time\'])/(b[\'time\']-a[\'time\'])\n    return {\'time\':time,\'bones\':[interpolate(x,y,t) for x,y in zip(a[\'bones\'],b[\'bones\'])],\n            \'actors\':[interpolate(x,y,t) for x,y in zip(a[\'actors\'],b[\'actors\'])],\'visible\':list(a[\'visible\'])}\n\n\ndef read_json(path):\n    try: return json.loads(core.read_limited(path).decode(\'utf-8\'),object_pairs_hook=core._no_duplicate_keys,parse_constant=core._bad_constant)\n    except (ValueError,UnicodeDecodeError) as exc: raise AssetError(f\'invalid manifest: {exc}\') from exc\n\n\ndef compile_manifest(manifest_path, output_prefix=None, *, base_color_only=False, single_sided_materials=False, force=False):\n    manifest_path=Path(manifest_path); manifest=read_json(manifest_path)\n    require(isinstance(manifest,dict) and manifest.get(\'schema\')==SCHEMA,\'invalid export manifest schema\')\n    def load(path):\n        require(isinstance(path,str) and path,\'GLB path must be a nonempty string\')\n        return Scene(*core.parse_glb(core.read_limited(manifest_path.parent/path)),base_color_only=base_color_only,single_sided_materials=single_sided_materials,armature=manifest.get(\'armature\'),rigid_armatures=manifest.get(\'rigid_armatures\',[]))\n    rest=load(manifest.get(\'rest_glb\')); names=manifest.get(\'actors\')\n    require(isinstance(names,list) and len(names)<=256,\'actors must be an array with at most 256 names\')\n    for warning in rest.warnings+rest.skin.warnings: print(warning,file=sys.stderr)\n    vrs,vrm,actors=rest.companions(names); clips=[]; total=0\n    rest_rigid=[rest.rigid_signature(a[\'name\']) for a in actors]\n    specifications=objects(manifest.get(\'clips\'),\'clips\'); require(1<=len(specifications)<=128,\'need 1..128 clips\')\n    for spec in specifications:\n        name=string(spec.get(\'name\'),\'clip name\'); require(type(spec.get(\'loop\')) is bool,\'clip loop must be boolean\')\n        scene=load(spec.get(\'path\'))\n        require([(b[0],b[1]) for b in scene.skin.bones]==[(b[0],b[1]) for b in rest.skin.bones],\'clip skeleton does not match rest skeleton\')\n        for a,b in zip(scene.skin.bones,rest.skin.bones):\n            require(max(abs(x-y) for x,y in zip(a[3],b[3]))<=2e-5,\'clip inverse binds differ from authoritative rest export\')\n        require(scene.skin.output_meshes==rest.skin.output_meshes,\'clip changes deform geometry/materials; runtime mesh companions are immutable\')\n        for actor,expected in zip(actors,rest_rigid):\n            actual=scene.rigid_signature(actor[\'name\'])\n            require(actual[0]==expected[0],\'clip changes rigid geometry/materials; runtime mesh companions are immutable\')\n            if actual[1] is None or expected[1] is None:\n                require(actual[1]==expected[1],\'clip changes rigid skin binding\')\n            else:\n                require(actual[1][0]==expected[1][0] and max(abs(x-y) for x,y in zip(actual[1][1],expected[1][1]))<=2e-5,\'clip changes rigid joint/inverse bind\')\n        times=spec.get(\'times\'); require(isinstance(times,list) and 1<=len(times)<=65536,\'clip times need 1..65536 samples\')\n        times=[core.f32(number(t,\'sample time\')) for t in times]\n        require(times[0]==0 and all(b>a for a,b in zip(times,times[1:])),\'clip times must start at zero and strictly increase\')\n        total+=len(times)*(len(rest.joints)+len(actors)); require(total<=MAX_TRANSFORMS,\'aggregate transform budget exceeded\')\n        duration=times[-1]\n        for ts,_,_ in scene.channels.values():\n            require(abs(ts[0])<=2e-5 and abs(ts[-1]-duration)<=max(2e-5,duration*2e-6), \'baked channels must cover complete normalized clip range\')\n        require(scene.channels or len(times)==1,\'animated clip has no sampled channels\')\n        visibility=spec.get(\'visibility\',[[1]*len(actors) for _ in times])\n        require(isinstance(visibility,list) and len(visibility)==len(times),\'visibility frame count mismatch\')\n        indices=[scene.actor_node(scene.node_named(a[\'name\'])) for a in actors]; frames=[]\n        for time,visible in zip(times,visibility):\n            require(isinstance(visible,list) and len(visible)==len(actors) and all(type(v) is int and v in (0,1) for v in visible),\'invalid visibility samples\')\n            bones,globals_=scene.sample(time)\n            frames.append({\'time\':time,\'bones\':bones,\'actors\':[decompose(globals_[i]) for i in indices],\'visible\':visible})\n        clips.append({\'name\':name,\'loop\':spec[\'loop\'],\'frames\':frames})\n    vra=encode_vra(rest.skin.bones,actors,clips,vrs,vrm)\n    if output_prefix is not None:\n        prefix=Path(output_prefix); outputs=[(prefix.with_suffix(e),d) for e,d in [(\'.vrs\',vrs),(\'.vrm\',vrm),(\'.vra\',vra)]]\n        require(all(force or not p.exists() for p,_ in outputs),\'output already exists; use --force\')\n        for p,d in outputs: core.atomic_write(p,d,force)\n    return vrs,vrm,vra\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__); commands=parser.add_subparsers(dest=\'command\',required=True)\n    pack=commands.add_parser(\'pack\'); pack.add_argument(\'manifest\',type=Path); pack.add_argument(\'output_prefix\',type=Path)\n    pack.add_argument(\'--base-color-only\',action=\'store_true\'); pack.add_argument(\'--single-sided-materials\',action=\'store_true\'); pack.add_argument(\'--force\',action=\'store_true\')\n    inspect=commands.add_parser(\'inspect\'); inspect.add_argument(\'input\',type=Path)\n    inspect.add_argument(\'--skin\',type=Path); inspect.add_argument(\'--mesh\',type=Path)\n    args=parser.parse_args(argv)\n    try:\n        if args.command==\'pack\':\n            vrs,vrm,vra=compile_manifest(args.manifest,args.output_prefix,base_color_only=args.base_color_only,single_sided_materials=args.single_sided_materials,force=args.force)\n            result=decode_vra(vra,vrs=vrs,vrm=vrm)\n        else: result=decode_vra(core.read_limited(args.input),vrs=core.read_limited(args.skin) if args.skin else None,vrm=core.read_limited(args.mesh) if args.mesh else None)\n        print(json.dumps({\'format\':MAGIC.decode(),\'bones\':len(result[\'bones\']),\'actors\':len(result[\'actors\']),\n                          \'clips\':[{\'name\':c[\'name\'],\'frames\':len(c[\'frames\']),\'duration\':c[\'frames\'][-1][\'time\'],\'loop\':c[\'loop\']} for c in result[\'clips\']]},indent=2))\n        return 0\n    except (AssetError,OSError,OverflowError,RecursionError,struct.error,UnicodeEncodeError) as exc:\n        print(f\'vrview: {exc}\',file=sys.stderr); return 1\n\n\nif __name__==\'__main__\': sys.exit(main())\n\n', 'review': '"""Rust Duty offline animation diagnostics. No artistic approval or rendering."""\nimport hashlib, io, json, math, platform, stat, zipfile\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nimport vrview\n\nMAX_INPUT = 128 * 1024 * 1024\nALLOWED = {\'.vra\', \'.vrs\', \'.vrm\', \'.json\', \'.png\', \'.jpg\', \'.jpeg\'}\n\ndef digest(data): return hashlib.sha256(data).hexdigest()\n\ndef safe_inputs(data):\n    """Read ZIP into memory, never extract or execute its contents."""\n    if len(data) > MAX_INPUT: raise ValueError(\'ZIP exceeds 128 MiB\')\n    result = {}\n    with zipfile.ZipFile(io.BytesIO(data)) as z:\n        infos = z.infolist()\n        if len(infos) > 512: raise ValueError(\'Too many ZIP members\')\n        if sum(i.file_size for i in infos) > 256 * 1024 * 1024:\n            raise ValueError(\'Expanded ZIP exceeds 256 MiB\')\n        for i in infos:\n            p = PurePosixPath(i.filename)\n            if (p.is_absolute() or \'..\' in p.parts or \'\\\\\' in i.filename or \':\' in i.filename\n                or stat.S_ISLNK(i.external_attr >> 16) or i.flag_bits & 1):\n                raise ValueError(\'Unsafe/encrypted ZIP member\')\n            if i.is_dir(): continue\n            if p.suffix not in ALLOWED: raise ValueError(\'Unsupported ZIP member: \' + i.filename)\n            if i.filename in result or i.file_size > MAX_INPUT: raise ValueError(\'Duplicate/oversize input\')\n            with z.open(i) as f: b = f.read(MAX_INPUT + 1)\n            if len(b) != i.file_size or len(b) > MAX_INPUT: raise ValueError(\'Input size mismatch\')\n            result[i.filename] = b\n    return result\n\ndef strict_json(data):\n    def reject(x): raise ValueError(\'Nonfinite JSON: \' + x)\n    def unique(pairs):\n        d = {}\n        for k,v in pairs:\n            if k in d: raise ValueError(\'Duplicate JSON key\')\n            d[k] = v\n        return d\n    def finite_float(x):\n        f=float(x)\n        if not math.isfinite(f): raise ValueError(\'Nonfinite JSON number\')\n        return f\n    return json.loads(data, parse_constant=reject, parse_float=finite_float, object_pairs_hook=unique)\n\ndef angle(a, b):\n    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)\n    a = a / np.linalg.norm(a, axis=-1, keepdims=True)\n    b = b / np.linalg.norm(b, axis=-1, keepdims=True)\n    # atan2 is stable for near-identical rotations and invariant to quaternion sign.\n    sign = np.where(np.sum(a*b, axis=-1, keepdims=True) < 0, -1., 1.)\n    b = b * sign\n    return np.degrees(4 * np.arctan2(np.linalg.norm(a-b, axis=-1), np.linalg.norm(a+b, axis=-1)))\n\ndef endpoint(a,b):\n    a=np.asarray(a); b=np.asarray(b)\n    if not a.size: return {\'translation_max\': None, \'rotation_degrees_max\': None, \'scale_max\': None}\n    return {\'translation_max\':float(np.linalg.norm(a[:,:3]-b[:,:3],axis=1).max()),\n            \'rotation_degrees_max\':float(angle(a[:,3:7],b[:,3:7]).max()),\n            \'scale_max\':float(np.abs(a[:,7:]-b[:,7:]).max())}\n\ndef channels(clip, kind): return np.asarray([f[kind] for f in clip[\'frames\']], dtype=np.float64)\n\ndef analyze_pack(data, skin=None, mesh=None, transitions=()):\n    pack=vrview.decode_vra(data, vrs=skin, vrm=mesh)\n    result={\'sha256\':digest(data), \'status\':\'diagnostic\',\n        \'coordinates\':\'glTF Y-up; bone TRS parent-local, actors scene-global; runtime Ry(pi) not applied\',\n        \'units\':\'export translation units; do not assume metres without source declaration\',\n        \'binding\':{\'skin_structure_crc_skeleton_checked\':skin is not None,\n                   \'rigid_structure_crc_assignment_checked\':mesh is not None,\n                   \'skin_deformation_contact_or_runtime_verified\':False},\n        \'clips\':[], \'transitions\':[]}\n    for c in pack[\'clips\']:\n        times=np.asarray([f[\'time\'] for f in c[\'frames\']]); dt=np.diff(times)\n        row={\'name\':c[\'name\'],\'frames\':len(times),\'duration_seconds\':float(times[-1]),\'loop\':c[\'loop\']}\n        for kind in (\'bones\',\'actors\'):\n            x=channels(c,kind)\n            values={\'endpoint_gap\': endpoint(x[-1],x[0]) if c[\'loop\'] else None}\n            if len(times)>1 and x.size:\n                speed=np.linalg.norm(np.diff(x[:,:,:3],axis=0),axis=2)/dt[:,None]\n                angular=angle(x[:-1,:,3:7],x[1:,:,3:7])/dt[:,None]\n                values.update(maximum_sampled_translation_speed=float(speed.max()),maximum_sampled_shortest_arc_rotation_deg_s=float(angular.max()))\n            row[kind]=values\n        v=np.asarray([f[\'visible\'] for f in c[\'frames\']])\n        row[\'visibility_change_count\']=int(np.count_nonzero(np.diff(v,axis=0))) if len(times)>1 else 0\n        result[\'clips\'].append(row)\n    lookup={c[\'name\']:c for c in pack[\'clips\']}\n    for pair in transitions:\n        if not isinstance(pair,(list,tuple)) or len(pair)!=2 or any(n not in lookup for n in pair):\n            raise ValueError(\'Transitions must name two existing clips in this pack\')\n        a,b=(lookup[n] for n in pair); row={\'from\':pair[0],\'to\':pair[1], \'scope\':\'unblended end-to-start; runtime blend is not modeled\'}\n        for kind in (\'bones\',\'actors\'): row[kind]=endpoint(a[\'frames\'][-1][kind],b[\'frames\'][0][kind])\n        row[\'visibility_mismatch_count\']=sum(x!=y for x,y in zip(a[\'frames\'][-1][\'visible\'],b[\'frames\'][0][\'visible\']))\n        result[\'transitions\'].append(row)\n    return pack,result\n\ndef plot_motion(pack, clip_name, kind, index, output):\n    import matplotlib.pyplot as plt\n    c=next(c for c in pack[\'clips\'] if c[\'name\']==clip_name)\n    x=channels(c,kind); t=np.array([f[\'time\'] for f in c[\'frames\']])\n    if kind not in (\'bones\',\'actors\') or not 0<=index<x.shape[1]: raise ValueError(\'Invalid channel\')\n    fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)\n    for k,label in enumerate(\'XYZ\'): axes[0].plot(t,x[:,index,k],label=label)\n    axes[0].set_ylabel(\'Translation (export units)\'); axes[0].legend()\n    if len(t)>1:\n        axes[1].plot((t[1:]+t[:-1])/2,np.linalg.norm(np.diff(x[:,index,:3],axis=0),axis=1)/np.diff(t))\n        axes[2].plot((t[1:]+t[:-1])/2,angle(x[:-1,index,3:7],x[1:,index,3:7])/np.diff(t))\n    axes[1].set_ylabel(\'Translation units / s\'); axes[2].set_ylabel(\'Rotation deg / s\'); axes[2].set_xlabel(\'Clip seconds\')\n    space=\'parent-local\' if kind==\'bones\' else \'scene-global\'\n    fig.suptitle(f\'{clip_name} / {kind}[{index}] / {space} / diagnostic\')\n    fig.tight_layout(); fig.savefig(output,dpi=130); plt.close(fig)\n\ndef contact_report(doc, use_cuda=False):\n    """Explicit corresponding evaluated points only, not nearest mesh or inferred anatomy."""\n    if doc.get(\'schema\')!=\'rust-duty-contact-samples/v1\' or doc.get(\'units\')!=\'metres\':\n        raise ValueError(\'Expected contact schema with metres\')\n    if not doc.get(\'source_sha256\') or len(doc[\'source_sha256\'])!=64 or any(c not in \'0123456789abcdef\' for c in doc[\'source_sha256\']):\n        raise ValueError(\'Require source artifact SHA256\')\n    if doc.get(\'space\')!=\'evaluated-world\' or not isinstance(doc.get(\'pairs\'),list) or not 1<=len(doc[\'pairs\'])<=64:\n        raise ValueError(\'Require evaluated-world points and 1..64 named pairs\')\n    report={\'status\':\'diagnostic\',\'source_sha256\':doc[\'source_sha256\'],\'source_hash_verified\':False,\n        \'backend\':\'numpy-float64\',\'pairs\':[], \'limitation\':\'Point separation only. No surface penetration, finger orientation, anatomical or artistic approval.\'}\n    for p in doc[\'pairs\']:\n        if not isinstance(p.get(\'name\'),str) or not p[\'name\']: raise ValueError(\'Pair needs name\')\n        t=np.asarray(p[\'time_seconds\'],dtype=np.float64); a=np.asarray(p[\'a\'],dtype=np.float64); b=np.asarray(p[\'b\'],dtype=np.float64)\n        mask=np.asarray(p[\'observed\'])\n        if (t.ndim!=1 or not 1<=len(t)<=200000 or a.shape!=(len(t),3) or b.shape!=a.shape\n            or mask.shape!=t.shape or mask.dtype!=np.bool_ or not np.isfinite(t).all()\n            or not np.isfinite(a).all() or not np.isfinite(b).all() or np.any(np.abs(a)>10000) or np.any(np.abs(b)>10000) or np.any(np.diff(t)<=0)):\n            raise ValueError(\'Invalid point samples/time/observability\')\n        cpu=np.linalg.norm(a-b,axis=1); distances=cpu\n        if use_cuda:\n            import torch\n            if not torch.cuda.is_available(): raise RuntimeError(\'CUDA requested but unavailable; choose CPU explicitly\')\n            with torch.no_grad():\n                distances=torch.linalg.vector_norm(torch.as_tensor(a,device=\'cuda\',dtype=torch.float64)-torch.as_tensor(b,device=\'cuda\',dtype=torch.float64),dim=1).cpu().numpy()\n            if not np.allclose(distances,cpu,rtol=1e-9,atol=1e-10): raise ValueError(\'CPU/CUDA disagreement\')\n            report[\'backend\']=\'torch-cuda-float64; CPU equivalence checked (rtol=1e-9, atol=1e-10)\'\n        observed=distances[mask]\n        report[\'pairs\'].append({\'name\':p[\'name\'],\'samples\':len(t),\'observed_samples\':int(mask.sum()),\n            \'coverage\':float(mask.mean()),\'max_observed_separation_m\':float(observed.max()) if len(observed) else None,\n            \'rms_observed_separation_m\':float(np.sqrt(np.mean(observed**2))) if len(observed) else None})\n    return report\n\ndef compare_images(a_bytes,b_bytes,output,source_label,candidate_label):\n    from PIL import Image, ImageDraw\n    images=[]\n    for blob in (a_bytes,b_bytes):\n        if len(blob)>32*1024*1024: raise ValueError(\'Image too large\')\n        im=Image.open(io.BytesIO(blob))\n        if im.width*im.height>16_000_000: raise ValueError(\'Image dimensions too large\')\n        im.load(); im=im.convert(\'RGB\'); im.thumbnail((1200,900)); images.append(im)\n    # Separate panels, no registration, stretch, crop, or automatic visual score.\n    canvas=Image.new(\'RGB\',(sum(i.width for i in images),max(i.height for i in images)+60),\'white\')\n    d=ImageDraw.Draw(canvas); x=0\n    for im,label in zip(images,(source_label,candidate_label)):\n        canvas.paste(im,(x,60)); d.text((x+8,8),label,fill=\'black\'); x+=im.width\n    canvas.save(output)\n\ndef verify_preserved_archive(archive, output, downloaded, expected_hash, expected_outputs):\n    """Validate a browser-returned saved copy and reject stale/changed output before teardown."""\n    archive=Path(archive); output=Path(output)\n    if not isinstance(downloaded,bytes) or digest(downloaded)!=expected_hash:\n        raise ValueError(\'Downloaded copy SHA256 mismatch; runtime retained\')\n    current={p.name:digest(p.read_bytes()) for p in output.iterdir() if p.is_file()}\n    if any(p.is_dir() for p in output.iterdir()) or current!=expected_outputs:\n        raise ValueError(\'Outputs changed after export; export and preserve a new ZIP first\')\n    if digest(archive.read_bytes())!=expected_hash:\n        raise ValueError(\'Local ZIP changed after export; runtime retained\')\n    with zipfile.ZipFile(io.BytesIO(downloaded)) as z:\n        if z.testzip() is not None or set(z.namelist())!=set(expected_outputs):\n            raise ValueError(\'Invalid result ZIP; runtime retained\')\n        if any(digest(z.read(n))!=sha for n,sha in expected_outputs.items()):\n            raise ValueError(\'Result contents differ; runtime retained\')\n    return {\'sha256\':expected_hash,\'bytes\':len(downloaded),\'preservation\':\'downloaded copy re-uploaded and byte-verified\'}\n'}
CODE_HASHES = {name: hashlib.sha256(s.encode()).hexdigest() for name,s in SOURCES.items()}
for name, source in SOURCES.items():
    module=types.ModuleType(name); module.__file__='<embedded:'+name+'>'
    sys.modules[name]=module; exec(compile(source,module.__file__,'exec'),module.__dict__)
import review
import numpy as np
from IPython.display import display, Image
OUTPUT=Path(tempfile.mkdtemp(prefix='rust-duty-review-'))
print('Output folder:', OUTPUT)
print('Python:',platform.python_version())
try:
    import torch
    print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('Actual GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch unavailable. CPU review works. No GPU package is installed automatically.')


## Inputs
ZIPs are read into bounded memory without extraction. Supported files only; paths, symlinks, duplicate names, encrypted files and oversize archives are rejected. Lowercase extensions only; plain `.vra`, not `.gz`.

Each pack may have same-prefix companions (`ads/asset.vra`, `ads/asset.vrs`, `ads/asset.vrm`). Companion assets are optional: omit anything private. Reports contain filenames/hashes/measurements, not copies of uploaded assets.


In [ ]:
DEMO = True # Change to False for your own authorized review ZIP.
if DEMO:
    import struct, zlib
    name=b'demo_loop'
    payload=struct.pack('<IIIH',0,0,1,4)+b'root'+struct.pack('<iII',-1,0,1)
    payload+=struct.pack('<H',len(name))+name+struct.pack('<II',1,3)
    for t,x in [(0.,0.),(.5,.01),(1.,0.)]:
        payload+=struct.pack('<f10f',t,x,0,0,0,0,0,1,1,1,1)
    demo=struct.pack('<8sIIII',b'VRANIM01',1,len(payload),zlib.crc32(payload),0)+payload
    INPUTS={'demo.vra':demo}
    INPUT_ORIGIN='synthetic diagnostic fixture, not game or approved motion'
else:
    from google.colab import files
    upload=files.upload()
    if len(upload)!=1 or not next(iter(upload)).lower().endswith('.zip'):
        raise ValueError('Select exactly one authorized review ZIP')
    INPUTS=review.safe_inputs(next(iter(upload.values())))
    INPUT_ORIGIN='user-selected ZIP; ownership and source provenance not independently verified'
print('Available packs:',[p for p in INPUTS if p.endswith('.vra')])


In [ ]:
PACK = 'demo.vra' # Use an exact available pack path.
TRANSITIONS = [] # Explicit [(from_clip, to_clip), ...], same pack only.
if PACK not in INPUTS: raise ValueError('Select an available PACK above')
OUTPUT=Path(tempfile.mkdtemp(prefix='rust-duty-review-')) # Fresh per pack review; no stale evidence
cuda_smoke={'status':'not run','scope':'synthetic numerical smoke, not game animation approval'}
contact={'status':'not measured','reason':'No evaluated world-space contact samples supplied'}
visual={'status':'not reviewed','artistic_approval':False}
prefix=PACK[:-4]
pack,report=review.analyze_pack(INPUTS[PACK],INPUTS.get(prefix+'.vrs'),INPUTS.get(prefix+'.vrm'),TRANSITIONS)
print(json.dumps(report,indent=2))
print('Bone channels:',list(enumerate(b[0] for b in pack['bones'])))
print('Actor channels:',list(enumerate(a['name'] for a in pack['actors'])))


In [ ]:
CLIP = pack['clips'][0]['name']
CHANNEL_KIND = 'bones' # 'bones' or 'actors'
CHANNEL_INDEX = 0
review.plot_motion(pack,CLIP,CHANNEL_KIND,CHANNEL_INDEX,OUTPUT/'motion.png')
display(Image(filename=str(OUTPUT/'motion.png')))


## Optional evaluated contact telemetry
Set CONTACT_JSON to a uploaded JSON name. Required schema:
`{"schema":"rust-duty-contact-samples/v1","units":"metres","space":"evaluated-world","source_sha256":"64 lowercase hex characters","pairs":[{"name":"index pad to fixed magazine anchor","time_seconds":[0,0.25],"a":[[0,0,0],[0,0,0]],"b":[[0,0,0],[0.001,0,0]],"observed":[true,false]}]}`

Coordinates are bounded to +/-10000 metres per axis. Supply evaluated corresponding world-space points from a known exported source, with named physical anchors. Do not substitute raw local bones for contact, invent landmarks or infer occluded samples. Sample switch neighborhoods and quarter frames in the authoring/export workflow. This tool only measures supplied point separation and observable coverage. It does not establish surface clearance, penetration, grip orientation or anatomy. If source bytes are absent, source hash remains declared, unverified.

CUDA is optional using Colab's existing PyTorch. Small arrays are usually faster/cheaper on CPU. Enabling CUDA records the actual backend and verifies against float64 CPU results (rtol 1e-9, atol 1e-10); lack of CUDA is an explicit error rather than a silent fallback. No speedup or A100 execution is claimed until this cell succeeds.


In [ ]:
CONTACT_JSON = '' # Empty means missing, not passed.
USE_CUDA_CONTACT = False
contact={'status':'not measured','reason':'No evaluated world-space contact samples supplied'}
if CONTACT_JSON:
    doc=review.strict_json(INPUTS[CONTACT_JSON])
    contact=review.contact_report(doc,use_cuda=USE_CUDA_CONTACT)
    contact['source_hash_verified']=doc['source_sha256'] in [review.digest(b) for b in INPUTS.values()]
print(json.dumps(contact,indent=2))


## Optional reference comparison
Use unwarped stills from the actual reference and the same candidate time. Fill in exact source frame/PTS and candidate sample time below. Images are separately aspect-preserving downscaled for display, never registered, cropped or stretched. No similarity score is computed. Inspect silhouette, wrist/finger anatomy, contact, camera and visibility yourself. This is a still comparison, not a complete clip review. No video decoding/timing is inferred.


In [ ]:
(OUTPUT/'comparison.png').unlink(missing_ok=True)
REFERENCE_IMAGE = ''
CANDIDATE_IMAGE = ''
REFERENCE_FRAME_AND_PTS = '' # Example: frame 123, PTS 4100/1000 seconds; observed, not inferred
CANDIDATE_SAMPLE_TIME = '' # Example: 4.1 seconds
visual={'status':'not reviewed','artistic_approval':False}
if REFERENCE_IMAGE or CANDIDATE_IMAGE:
    if not all((REFERENCE_IMAGE,CANDIDATE_IMAGE,REFERENCE_FRAME_AND_PTS,CANDIDATE_SAMPLE_TIME)):
        raise ValueError('Specify both images and exact frame/time labels')
    review.compare_images(INPUTS[REFERENCE_IMAGE],INPUTS[CANDIDATE_IMAGE],OUTPUT/'comparison.png',
                          'Reference: '+REFERENCE_FRAME_AND_PTS,'Candidate: '+CANDIDATE_SAMPLE_TIME)
    display(Image(filename=str(OUTPUT/'comparison.png')))
    visual.update(status='paired stills prepared; human inspection required',reference=REFERENCE_IMAGE,
                  candidate=CANDIDATE_IMAGE,reference_frame_pts=REFERENCE_FRAME_AND_PTS,candidate_time=CANDIDATE_SAMPLE_TIME)


In [ ]:
# Optional one-shot synthetic GPU smoke. Set True for this A100 batch.
RUN_CUDA_SMOKE = False
cuda_smoke={'status':'not run','scope':'synthetic numerical smoke, not game animation approval'}
(OUTPUT/'cuda-smoke.json').unlink(missing_ok=True)
if RUN_CUDA_SMOKE:
    import time, torch
    if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable; no GPU execution claimed')
    device=torch.cuda.get_device_name(0)
    if 'A100' not in device: raise RuntimeError('This batch requests A100; actual device is '+device)
    n=4096
    t=np.arange(n,dtype=np.float64)/240.0
    a=np.column_stack((np.sin(t)*.1,np.cos(t)*.1,t*.001))
    b=a+np.array([.001,-.002,.0005],dtype=np.float64)
    synthetic_bytes=a.tobytes()+b.tobytes()+t.tobytes()
    doc={'schema':'rust-duty-contact-samples/v1','units':'metres','space':'evaluated-world',
         'source_sha256':review.digest(synthetic_bytes),
         'pairs':[{'name':'synthetic points only','time_seconds':t.tolist(),
                   'a':a.tolist(),'b':b.tolist(),'observed':[True]*n}]}
    torch.cuda.synchronize(); started=time.perf_counter()
    checked=review.contact_report(doc,use_cuda=True)
    torch.cuda.synchronize()
    cuda_smoke={'status':'passed','scope':'synthetic numerical smoke, not game animation approval',
                'device':device,'torch':torch.__version__,'cuda_runtime':torch.version.cuda,
                'samples':n,'elapsed_seconds_including_cpu_check':time.perf_counter()-started,
                'cpu_parity_rtol':1e-9,'cpu_parity_atol':1e-10,'result':checked}
    (OUTPUT/'cuda-smoke.json').write_text(json.dumps(cuda_smoke,indent=2,allow_nan=False))
print(json.dumps(cuda_smoke,indent=2))


In [ ]:
# Reproducible, evidence-scoped result; no raw uploaded assets included.
if not (OUTPUT/'motion.png').exists(): raise ValueError('Run the motion plot for this pack before exporting')
versions={p:importlib.metadata.version(p) for p in ('numpy','matplotlib','Pillow')}
try: versions['torch']=importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError: pass
result={'schema':'rust-duty-colab-review/v1','status':'diagnostic','artistic_approval':False,
        'input_origin':INPUT_ORIGIN,'inputs':{n:{'sha256':review.digest(b),'bytes':len(b)} for n,b in INPUTS.items()},
        'code_sha256':CODE_HASHES,'python':platform.python_version(),'dependencies':versions,
        'selection':{'pack':PACK,'clip':CLIP,'channel_kind':CHANNEL_KIND,'channel_index':CHANNEL_INDEX,'transitions':TRANSITIONS},
        'pack':report,'contact':contact,'visual':visual,
        'cuda_smoke':globals().get('cuda_smoke',{'status':'not run'}),
        'remaining_gates':['evaluated skin and full-motion visual inspection','source timing/camera verification',
                           'Blender evaluated bake/export/reimport parity','target game runtime verification']}
(OUTPUT/'report.json').write_text(json.dumps(result,indent=2,allow_nan=False))
archive=OUTPUT.parent/(OUTPUT.name+'.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.iterdir()): z.write(path,path.name)
EXPORTED_OUTPUT_HASHES={p.name:review.digest(p.read_bytes()) for p in OUTPUT.iterdir() if p.is_file()}
ARCHIVE_SHA256=review.digest(archive.read_bytes())
print('Saved:',archive,'SHA256:',ARCHIVE_SHA256)
from google.colab import files
files.download(str(archive))


## Preserve, then release this batch runtime
This notebook runs one bounded batch. Connect the selected A100 only when ready; run the optional synthetic CUDA smoke before the export cell. No keepalive, idle loop, automatic reconnect or recurring GPU reservation.

The export cell initiates a browser download; it does **not** prove the ZIP reached durable storage. Wait for the browser's download to finish, retain that file outside Colab, then run the guarded final cell with `CONFIRM_TEARDOWN=True`. Select the downloaded ZIP when prompted. The cell verifies the exact ZIP bytes and checks that outputs have not changed since export before requesting `runtime.unassign()`. Cancellation, mismatches and errors retain the runtime so you can recover the output. Only files in this review's OUTPUT directory are covered; preserve any unrelated runtime work separately before confirming.

After unassignment, verify the notebook disconnects AND its entry disappears from **Runtime → Manage sessions**. Disconnection alone is not proof of resource release. If it remains, use **Runtime → Disconnect and delete runtime** after preserving output; do not reconnect to check. There is no billing-finality or immediate host-deletion guarantee from a cell, and Colab can independently terminate sessions.

Official sources: [runtime.unassign implementation](https://github.com/googlecolab/colabtools/blob/main/google/colab/runtime.py), [asynchronous files.download implementation](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py), [Colab FAQ](https://research.google.com/colaboratory/faq.html). No new Drive permission or external upload destination is needed. Locally tested guards are not a claim of A100 or actual Colab teardown execution.


In [ ]:
import io, zipfile
from pathlib import Path
from review import digest
def verify_preserved_archive(archive, output, downloaded, expected_hash, expected_outputs):
    """Validate a browser-returned saved copy and reject stale/changed output before teardown."""
    archive=Path(archive); output=Path(output)
    if not isinstance(downloaded,bytes) or digest(downloaded)!=expected_hash:
        raise ValueError('Downloaded copy SHA256 mismatch; runtime retained')
    current={p.name:digest(p.read_bytes()) for p in output.iterdir() if p.is_file()}
    if any(p.is_dir() for p in output.iterdir()) or current!=expected_outputs:
        raise ValueError('Outputs changed after export; export and preserve a new ZIP first')
    if digest(archive.read_bytes())!=expected_hash:
        raise ValueError('Local ZIP changed after export; runtime retained')
    with zipfile.ZipFile(io.BytesIO(downloaded)) as z:
        if z.testzip() is not None or set(z.namelist())!=set(expected_outputs):
            raise ValueError('Invalid result ZIP; runtime retained')
        if any(digest(z.read(n))!=sha for n,sha in expected_outputs.items()):
            raise ValueError('Result contents differ; runtime retained')
    return {'sha256':expected_hash,'bytes':len(downloaded),'preservation':'downloaded copy re-uploaded and byte-verified'}

# Run separately AFTER the completed download is saved outside this runtime.
# Re-upload that downloaded ZIP as byte-level preservation evidence.
CONFIRM_TEARDOWN = False
if not CONFIRM_TEARDOWN:
    print('Runtime retained. Save the results ZIP, then set CONFIRM_TEARDOWN=True and run this cell.')
else:
    if 'ARCHIVE_SHA256' not in globals() or 'EXPORTED_OUTPUT_HASHES' not in globals():
        raise RuntimeError('Export the current results first; runtime retained')
    from google.colab import files, runtime
    print('Select the downloaded results ZIP. No teardown on cancel, error or mismatch.')
    saved_copy=files.upload()
    if len(saved_copy)!=1: raise ValueError('Select exactly one saved results ZIP; runtime retained')
    preservation=verify_preserved_archive(archive,OUTPUT,next(iter(saved_copy.values())),
                                                ARCHIVE_SHA256,EXPORTED_OUTPUT_HASHES)
    print('Verified saved copy:',preservation)
    print('Requesting runtime unassignment now. Verify removal under Runtime > Manage sessions.')
    runtime.unassign()
